# Agente de Triagem de Ordens de Serviço de Manutenção

## Entrega 1 — Caracterização do Caso e Agente Único

**Engenharia de Agentes e IA Agêntica** · Pós-graduação · PUC Minas · Prof. Paulo Moreira

| | |
|---|---|
| **Empresa (fictícia)** | Construtora Apollo S.A. |
| **Ambiente** | Google Colab |
| **SDK** | `google-genai` · `gemini-3.6-flash` (reservas: `gemini-3.5-flash` e `gemini-3.1-flash-lite`) · embeddings `gemini-embedding-001` |
| **Base de conhecimento** | Vector RAG com ChromaDB, local |
| **Ferramentas** | Tool Calling estruturado, integração direta (sem MCP) |
| **Repositório** | [github.com/diogo-hsf/apollo-agente-manutencao](https://github.com/diogo-hsf/apollo-agente-manutencao) |
| **Requisito externo** | Apenas a `GEMINI_API_KEY` |

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/diogo-hsf/apollo-agente-manutencao/blob/main/notebook/Apollo_Agente_Triagem_Manutencao.ipynb)

> **Como executar.** Cadastre a chave do Gemini nos *Secrets* do Colab com o nome `GEMINI_API_KEY` (ícone de chave na barra lateral) e use *Ambiente de execução → Executar tudo*. Na primeira execução, o Colab pede permissão para o notebook acessar o secret: é preciso conceder. Documentos e tabelas são baixados do repositório pelo próprio notebook.

> **Sobre a cota da API.** No nível gratuito, o `gemini-3.6-flash` permite cerca de 20 chamadas de geração por dia, e uma execução completa usa entre 12 e 16 delas — apenas as seções 7.8 e 8.3 chamam o modelo. Recomenda-se executar com a cota do dia ainda disponível. Se o modelo principal estiver sobrecarregado (erro 503) ou esgotar a cota, o notebook passa para um modelo de reserva (seção 3.4). Se nenhum modelo responder, o notebook não é interrompido: as triagens afetadas são registradas como `incompleta` ou "não executada", e os testes de comportamento indicam isso explicitamente.

> **Sobre os dados.** A Construtora Apollo S.A., seus documentos técnicos, os códigos de alarme TL e os códigos internos de peças APL são fictícios, criados para esta disciplina. Os modelos de equipamento são reais (Caterpillar 320, D9T e 924H; Mercedes-Benz Axor e 2726). Os manuais dos fabricantes foram usados apenas como referência factual — intervalos de manutenção, lógica de níveis de alerta e referências de peças —, sem reprodução de trechos.

## Sumário

1. Caracterização do caso
2. Desenho da solução
3. Configuração do ambiente
4. Dados operacionais
5. Base de conhecimento técnica (Vector RAG)
6. Ferramentas
7. O agente de triagem
8. Testes de comportamento
9. Conclusão

---
# 1. Caracterização do caso

## 1.1 A empresa

A **Construtora Apollo S.A.** executa obras de infraestrutura pesada — duplicação rodoviária, alteamento de barragem de mineração e terraplenagem industrial — com frota própria de escavadeiras, tratores de esteiras, carregadeiras de rodas e caminhões basculantes, distribuída entre as obras.

Nesse negócio, equipamento parado é custo direto. Uma escavadeira parada não só deixa de produzir: ela trava os caminhões que carregaria, a frente de serviço que dependia dela e o cronograma da obra. O indicador perseguido pela gestão de manutenção é a **disponibilidade física da frota**.

## 1.2 O problema de negócio

Quando um equipamento apresenta problema, o operador abre uma **ordem de serviço (OS)** de manutenção. Ela chega ao planejador de manutenção como um texto curto, escrito no campo e quase sempre incompleto:

- *"Alarme TL-214 no painel, lança e braço ficam lentos depois de umas duas horas de trabalho."*
- *"Caminhão fazendo barulho estranho na frente."*
- *"Vazamento de óleo no cilindro da caçamba da escavadeira."* — sem dizer qual escavadeira.

Antes que um mecânico vá ao equipamento, alguém precisa fazer a **triagem** dessa OS:

| Pergunta da triagem | Onde está a resposta hoje |
|---|---|
| O que esse alarme significa e qual o procedimento de diagnóstico? | Fichas técnicas e catálogo de alarmes, em PDF |
| A situação exige bloquear o equipamento? | Procedimento de segurança, em PDF |
| Essa falha já aconteceu antes nesse equipamento? | Histórico de manutenção, no sistema |
| A revisão preventiva está em dia? | Plano de manutenção e horímetro, no sistema |
| A peça provável está disponível? Em qual almoxarifado? | Estoque, no sistema |
| Há equipamento substituto se este parar? | Cadastro da frota, no sistema |
| Qual a prioridade? | Política de manutenção, cruzada com tudo acima |

Hoje essa triagem é manual. A informação está espalhada entre documentos e sistemas, e a busca leva minutos ou horas por OS. O custo aparece em quatro lugares:

- **Parada prolongada:** o mecânico só começa quando a triagem termina — ou começa sem ela.
- **Diagnóstico errado:** sem consultar o histórico, troca-se a mesma peça pela terceira vez sem atacar a causa raiz.
- **Espera por peça:** a falta da peça só é descoberta com o equipamento desmontado.
- **Risco de segurança:** um equipamento com falha que exige bloqueio continua operando porque ninguém consultou o procedimento a tempo.

## 1.3 Duas naturezas de informação

A triagem combina dois tipos de informação, e a solução trata cada um de forma diferente:

| | **Conhecimento técnico (regras)** | **Estado operacional (fatos)** |
|---|---|---|
| O que é | Significado de alarmes, procedimentos de diagnóstico, regras de segurança e de prioridade | Situação atual de cada equipamento, peça e obra |
| Onde vive | Fichas técnicas, catálogo de alarmes, procedimento de segurança, política de manutenção | Cadastro, histórico, horímetro e preventivas, estoque, frota |
| Muda com que frequência | Raramente, por revisão de documento | A cada OS, a cada movimentação de estoque |
| Exemplo | *"Três corretivas no mesmo sistema em 90 dias caracterizam falha recorrente"* | *"A EH-07 teve duas corretivas hidráulicas nos últimos 45 dias"* |
| Como o agente acessa | **Base de conhecimento (Vector RAG)** | **Ferramentas (Tool Calling)** |

## 1.4 O que vamos construir

Um **agente de triagem de OS de manutenção**. Para cada OS aberta, o agente investiga — consultando a documentação técnica e os sistemas conforme a necessidade *daquela* OS — e registra na **fila de manutenção** um parecer com:

| Saída | Pergunta que responde |
|---|---|
| Diagnóstico provável | O que provavelmente está acontecendo, com a fonte técnica que sustenta |
| Procedimento recomendado | Por onde o mecânico deve começar |
| Peças e disponibilidade | O que provavelmente será necessário e onde existe em estoque |
| Recorrência | A falha é nova ou repetida? |
| Situação da preventiva | A revisão está em dia, próxima ou vencida? |
| Prioridade | Com que urgência tratar, segundo a matriz da política de manutenção |
| Recomendação operacional | Liberar, liberar com restrição ou bloquear — **sempre como recomendação** |
| Pendências e perguntas | O que não foi possível verificar e o que precisa ser perguntado ao solicitante |

## 1.5 O que o agente **não** faz

- **Não executa** reparo nem altera nada nos sistemas de manutenção.
- **Não bloqueia nem libera** equipamentos — apenas recomenda; a decisão é de pessoa habilitada (PRO-SEG-001).
- **Não emite** requisição de compra de peças.
- **Não substitui** a inspeção presencial do mecânico.
- **Não inventa** diagnóstico. Se o relato for vago ou o equipamento não puder ser identificado, a saída correta é devolver a OS com perguntas.

## 1.6 Por que uma abordagem agêntica?

Pelo critério visto na Aula 2 — *"o fluxo é conhecido e determinável? Então um workflow pode bastar"* —, a resposta depende de o caminho da triagem ser previsível. Ele não é.

**O caminho da investigação muda a cada OS.** Quatro ordens do mesmo dia exigem caminhos diferentes:

| OS | Caminho de investigação adequado |
|---|---|
| *"EH-07 com alarme TL-214, lança e braço lentos depois de aquecer"* | Ficha técnica (TL-214) → histórico (a falha se repete?) → política (o que fazer com falha recorrente) → estoque |
| *"Caminhão fazendo barulho estranho na frente"* | Ficha técnica mostra que o ruído pode ser freio, suspensão ou direção → relato insuficiente → devolver com as perguntas certas |
| *"Trator perdendo força na rampa, fumaça escura no escapamento"* | Preventiva primeiro: revisão de filtros vencida é a hipótese principal; histórico confirma |
| *"Vazamento de óleo no cilindro da caçamba da escavadeira"* | Equipamento não identificado e há mais de uma escavadeira na obra → não é permitido deduzir → devolver |

Um workflow determinístico teria de prever cada combinação de sintoma, equipamento e situação — ou executaria **todas** as consultas em **toda** OS, gerando custo, ruído e respostas genéricas.

**A entrada é texto livre e incompleto.** O sintoma vem na linguagem do operador, com ou sem código de alarme. Interpretar *"fica lento depois que esquenta"* exige julgamento, não regra.

**É preciso cruzar regra e fato.** *"A ficha manda limpar o arrefecedor e trocar o filtro"* (regra) somado a *"isso já foi feito duas vezes neste mês"* (fato) muda a conclusão para *"falha recorrente: análise de causa raiz; repetir a troca não resolve"*.

**É preciso saber quando parar — e quando não concluir.** O agente decide quando já tem evidência suficiente e reconhece quando a informação disponível não sustenta um diagnóstico.

**O que continua determinístico.** Seguindo o princípio *aritmética no código, julgamento no modelo*, ficam em Python: horas desde a última preventiva, contagem de recorrência, saldo de estoque, validação de identificadores, detecção de gatilhos de segurança, limites de execução e validação do parecer.

**Por que um agente único.** A triagem é uma responsabilidade coesa, com um objetivo e um responsável. Ainda não há especialização que justifique dividir o trabalho entre vários agentes.

## 1.7 O ambiente do agente

```
═══════════════════════════ AMBIENTE ═══════════════════════════

              Operador / motorista do equipamento
                         │  abre uma OS
                         ▼
                ordens_servico.csv
                         │  percepção
                         ▼
   ┌───────────────────────────────────────────────────┐
   │        AGENTE DE TRIAGEM DE MANUTENÇÃO            │
   │                                                   │
   │   perceber()  → o que foi relatado?               │
   │   decidir()   → laço: decidir → agir → observar   │
   │   agir()      → registrar o parecer               │
   │                                                   │
   │   ┌───────────────────────────────────────────┐   │
   │   │ LLM (gemini-3.6-flash)                    │   │
   │   │ mecanismo de inferência                   │   │
   │   └───────────────────────────────────────────┘   │
   └───────────────────────────────────────────────────┘
          │                         ▲
          │ ação                    │ consultas (Tool Calling)
          ▼                         │
  fila_manutencao.csv     ┌─────────┴──────────────────────────┐
          │               │ Documentação técnica (Vector RAG)  │
          ▼               │ Equipamentos · Histórico           │
  Planejador de           │ Preventivas · Estoque · Frota      │
  manutenção / mecânico   └────────────────────────────────────┘

════════════════════════════════════════════════════════════════
```

## 1.8 Framework PEAS

| Elemento | Neste agente |
|---|---|
| **P**erformance | Diagnóstico fundamentado em fonte técnica recuperada; nenhum fato operacional afirmado sem consulta ao sistema; ferramentas escolhidas de acordo com a OS; devolução de OS com informação insuficiente; recomendação de segurança conservadora; conclusão dentro dos limites de execução |
| **E**nvironment | Ordens de serviço da Construtora Apollo, documentação técnica da frota e sistemas de manutenção, estoque e frota |
| **A**ctuators | Registro do parecer na fila de manutenção (`fila_manutencao.csv`) |
| **S**ensors | Leitura da OS e resultados das ferramentas: documentação técnica, cadastro, histórico, preventivas, estoque e frota |

Os **sensores são ativos**: o agente não apenas recebe a OS — ele decide o que mais precisa observar.

## 1.9 Propriedades do ambiente

| Propriedade | Classificação | Consequência para o projeto |
|---|---|---|
| Observável | **Parcialmente** | O relato é incompleto e o agente não vê a máquina; precisa de ferramentas e deve declarar incertezas |
| Agentes | **Único** | Só este agente atua; os humanos consomem o resultado |
| Determinismo | **Estocástico** | A mesma falha pode ter causas diferentes; o diagnóstico é sempre "provável" |
| Episódico | **Episódico por OS, com memória no histórico** | Cada OS é independente, mas o histórico de OS anteriores é informação decisiva |
| Estático | **Dinâmico** | Estoque e status mudam ao longo do dia; fatos são consultados no momento da triagem, nunca supostos |
| Conhecido | **Conhecido** | As regras estão documentadas; o desafio é localizá-las e aplicá-las |

## 1.10 Tipo de agente e nível de autonomia

**Tipo.** Agente **baseado em objetivo** — tem uma meta explícita (uma triagem fundamentada e acionável) e escolhe uma sequência de ações para atingi-la —, com raciocínio no padrão **ReAct**: decide, age (consulta), observa o resultado e decide de novo. A definição de prioridade tem um componente de **utilidade**, porque pondera segurança, criticidade e disponibilidade de substituto. A detecção de gatilhos de segurança é uma **camada reativa** em código, o que caracteriza um agente **híbrido**.

| Característica | Presente? | Por quê |
|---|---|---|
| Autonomia | Parcial | Decide sozinho o que investigar e o que concluir, mas apenas recomenda |
| Reatividade | Sim | É acionado pela abertura de uma OS |
| Proatividade | Parcial | Dentro da triagem, busca por iniciativa própria o que falta |
| Sociabilidade | Não | Agente único nesta entrega |

**Nível de autonomia** (matriz *"qual nível para cada tarefa"*, Aula 1):

| Decisão | Impacto | Nível adotado |
|---|---|---|
| Diagnóstico, procedimento, peças, prioridade | Médio e reversível — o mecânico confirma em campo | **Supervisionado**: o agente produz, o planejador revisa |
| Liberar ou bloquear o equipamento | Alto — segurança de pessoas | **Humano no loop, sempre**: o agente só recomenda, com viés conservador |

## 1.11 Escopo desta entrega

| Esta entrega contempla | Fica para as próximas |
|---|---|
| Agente único com laço de decisão, ação e observação (ReAct) | Orquestração com framework |
| Base de conhecimento técnica com Vector RAG | Sistema multiagente |
| Seis ferramentas com Tool Calling estruturado | Governança, auditoria e human-in-the-loop formal |
| Validação, tratamento de erros, limites e recuperação de falhas | Memória entre triagens |
| Testes de comportamento | |

---
# 2. Desenho da solução

## 2.1 Arquitetura

```
                 ordens_servico.csv
                         │
                         ▼
┌──────────────────────────────────────────────────────────────┐
│ perceber()                                     [código]      │
│ valida a OS · extrai códigos de alarme · resolve o           │
│ equipamento no cadastro · detecta gatilhos de segurança      │
└──────────────────────────────────────────────────────────────┘
                         │
                         ▼
┌──────────────────────────────────────────────────────────────┐
│ decidir()  laço ReAct                     [LLM + runtime]    │
│                                                              │
│  ┌──────────┐   function_call    ┌───────────────────────┐   │
│  │  Gemini  │ ─────────────────► │ Runtime               │   │
│  │ 3.6 Flash│                    │ · valida argumentos   │   │
│  │          │ ◄───────────────── │ · aplica limites      │   │
│  └──────────┘ function_response  │ · executa a tool      │   │
│        │                         │ · normaliza erros     │   │
│        │ emitir_parecer(...)     └───────────┬───────────┘   │
│        ▼                                     │               │
│  ┌──────────────────────────┐   ┌────────────▼────────────┐  │
│  │ Validação do parecer     │   │ Ferramentas             │  │
│  │ schema · domínio ·       │   │ RAG técnico · histórico │  │
│  │ guardrail de segurança   │   │ preventiva · estoque ·  │  │
│  └──────────────────────────┘   │ frota · equipamento     │  │
│                                 └─────────────────────────┘  │
└──────────────────────────────────────────────────────────────┘
                         │
                         ▼
┌──────────────────────────────────────────────────────────────┐
│ agir()                                         [código]      │
│ grava o parecer em fila_manutencao.csv                       │
└──────────────────────────────────────────────────────────────┘
```

Sem MCP, o notebook usa o cliente **síncrono** do `google-genai` e um **registro explícito** de ferramentas (`nome → função + schema de entrada`), como no slide de integração direta da Aula 3.

## 2.2 Divisão de responsabilidades

| Em Python (determinístico) | No modelo (julgamento) |
|---|---|
| Validar a OS e resolver o equipamento informado | Interpretar o relato em linguagem livre |
| Extrair códigos de alarme do texto (`TL-214`) | Decidir o que investigar e em que ordem |
| Calcular horas desde a última preventiva e a situação | Formular as perguntas à base técnica |
| Contar ocorrências do mesmo sistema em 90 dias | Cruzar regra técnica com fato operacional |
| Somar saldo de estoque por almoxarifado | Formar diagnóstico provável e prioridade |
| Detectar gatilhos críticos de segurança no relato | Decidir quando parar ou devolver a OS |
| Aplicar limites e validar o parecer | |

## 2.3 Base de conhecimento técnica

| Código | Documento | Aplicação |
|---|---|---|
| FTM-CAT320 | Ficha técnica de manutenção — Escavadeira CAT 320 | escavadeiras |
| FTM-CATD9T | Ficha técnica de manutenção — Trator de esteiras CAT D9T | tratores |
| FTM-CAT924H | Ficha técnica de manutenção — Carregadeira CAT 924H | carregadeiras |
| FTM-MBCAM | Ficha técnica de manutenção — Caminhões basculantes Mercedes-Benz | caminhões |
| CAT-ALM-001 | Catálogo de alarmes de telemetria (códigos TL) | geral |
| PRO-SEG-001 | Procedimento de segurança — bloqueio e impedimento de operação | geral |
| POL-MAN-001 | Política de manutenção da frota | geral |

Os PDFs têm seções numeradas, o que permite **chunking por seção** e citações no formato `FTM-CAT924H § 5.3`. Cada chunk recebe nos metadados a sua aplicação: a busca para uma escavadeira traz a ficha da escavadeira e os documentos gerais, nunca a ficha do caminhão.

A matriz de prioridade e a regra de recorrência ficam **no documento**, não no código: o agente precisa recuperar a regra antes de aplicá-la — o padrão "regra no documento, fato no sistema".

## 2.4 Ferramentas

| Ferramenta | Pergunta que responde | Fonte |
|---|---|---|
| `buscar_documentacao_tecnica` | O que diz a documentação? | Vector RAG |
| `consultar_equipamento` | Qual equipamento é este? | Cadastro |
| `consultar_historico_manutencao` | Já aconteceu antes? | Histórico de corretivas |
| `consultar_plano_preventiva` | A manutenção está em dia? | Plano + registro de preventivas |
| `consultar_estoque_pecas` | A peça existe? Onde? | Estoque |
| `consultar_frota_disponivel` | Há substituto se parar? | Cadastro |
| **`emitir_parecer`** | *(ferramenta terminal)* | — |

Todas as ferramentas devolvem um **status padronizado**: `success`, `not_found` (consulta válida, sem registro — ausência de informação, não erro), `invalid_arguments` (com a mensagem de correção) ou `error` (falha interna, sem detalhes técnicos expostos ao modelo).

**Ferramenta terminal.** O parecer final é entregue chamando `emitir_parecer`, cujos argumentos seguem o schema da saída. Isso dispensa uma chamada extra de finalização: o modelo conclui na mesma rodada em que decide parar.

## 2.5 Limites e robustez

| Mecanismo | Como funciona |
|---|---|
| Limite de rodadas do modelo e de chamadas de ferramenta por triagem | Ao atingir, o runtime força a finalização: só `emitir_parecer` fica disponível |
| Validação de entrada | Argumentos conferidos contra o schema antes de executar |
| Validação de retorno | Saída da ferramenta conferida contra o contrato antes de chegar ao modelo |
| Erros de API transitórios (429, 500, 503, 504) | Retry com backoff exponencial |
| Cota diária esgotada | Sem retry inútil: triagem marcada `incompleta`, evidências preservadas |
| Parecer inválido | Os problemas são registrados e a triagem vai para revisão humana obrigatória |
| Gatilho de segurança | Camada reativa em código impede a recomendação "liberar" |

## 2.6 Cota da API

O nível gratuito do Gemini permite cerca de **20 chamadas de geração por dia** por modelo. O agente pede na mesma rodada as consultas independentes entre si, e cada rodada conta como uma única chamada, qualquer que seja o número de ferramentas pedidas. A cota é contada por modelo: quando a do modelo principal se esgota, ou quando ele fica sobrecarregado, o notebook passa para um modelo de reserva. Quando a cota de todos os modelos se esgota, a triagem em andamento termina em contingência e as seguintes não são enviadas à API. Os testes de limites e de falhas usam um **modelo simulado**, sem custo de cota.

---
# 3. Configuração do ambiente

## 3.1 Dependências

Versões fixadas para que o notebook se comporte igual em qualquer execução. O `google-genai` segue a mesma versão usada nas aulas.

In [ ]:
%pip install -q "google-genai==2.25.0" "chromadb>=1.0,<2" "pypdf>=5,<7" "jsonschema>=4.23,<5"

In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import random
import re
import time
import unicodedata
import urllib.request
from dataclasses import dataclass, field
from datetime import date, datetime, timedelta
from functools import lru_cache
from pathlib import Path
from typing import Any, Callable

import chromadb
import httpx
import pandas as pd
from google import genai
from google.genai import errors, types
from jsonschema import Draft202012Validator
from pypdf import PdfReader

print("Bibliotecas importadas.")
print(f"google-genai {genai.__version__} · chromadb {chromadb.__version__} · pandas {pd.__version__}")

## 3.2 Parâmetros

Todos os parâmetros que mudam o comportamento do notebook ficam nesta célula.

`MODELOS` lista os modelos de geração em ordem de preferência. O principal é o `gemini-3.6-flash`; os demais são reservas, usados quando o anterior fica indisponível ou esgota a cota do dia (seção 3.4).

`DATA_REFERENCIA` é o "hoje" do caso. As janelas de histórico e o cálculo de recorrência usam essa data, e não a data real de execução — assim o resultado é o mesmo em qualquer dia em que o notebook for executado.

In [ ]:
# Repositório com documentos e tabelas (público; baixado via raw.githubusercontent.com)
REPOSITORIO_RAW = "https://raw.githubusercontent.com/diogo-hsf/apollo-agente-manutencao/main/"
PASTA_LOCAL = Path("apollo_dados")            # cópia local dos arquivos baixados

DATA_REFERENCIA = date(2026, 9, 28)           # "hoje" do caso

# Modelos de geração, em ordem de preferência: o primeiro é o principal; os demais são
# reservas para quando o anterior estiver sobrecarregado (503) ou com a cota do dia esgotada.
MODELOS = ["gemini-3.6-flash", "gemini-3.5-flash", "gemini-3.1-flash-lite"]
MODELO = MODELOS[0]
MODELO_EMBEDDING = "gemini-embedding-001"
DIMENSAO_EMBEDDING = 768

# Resiliência das chamadas à API
TENTATIVAS_POR_CHAMADA = 3                    # retry para erros transitórios
ESPERA_BASE_SEGUNDOS = 30.0                   # esperas de 30 s e 60 s, como nas Aulas 1 e 2
ESPERA_MAXIMA_ACEITAVEL = 90                  # retryDelay maior que isso = cota diária
TIMEOUT_CHAMADA_MS = 180_000                  # o modelo pode raciocinar por bastante tempo

print(f"Modelo principal: {MODELO} · reservas: {', '.join(MODELOS[1:])}")
print(f"Embeddings: {MODELO_EMBEDDING} ({DIMENSAO_EMBEDDING} dim)")
print(f"Data de referência do caso: {DATA_REFERENCIA:%d/%m/%Y}")

## 3.3 Chave da API e conexão

A chave é lida dos *Secrets* do Colab (`GEMINI_API_KEY`), de uma variável de ambiente ou, em último caso, digitada com entrada oculta. Ela nunca aparece no código.

Dois detalhes de configuração do cliente:

- **O retry interno do SDK é desligado** (`attempts=1`). O retry fica a cargo da camada da seção 3.4, que distingue erro transitório de cota diária esgotada. Com os dois ligados, um erro seria tentado de novo em dobro, gastando tempo à toa.
- **A conexão é verificada sem gastar cota.** `models.get` consulta os metadados de cada modelo da lista: confirma que a chave é válida e quais modelos estão disponíveis para ela, sem consumir nenhuma das ~20 chamadas de geração do dia. Modelos indisponíveis para a chave saem da lista.

In [ ]:
def obter_api_key() -> str:
    """Recupera a chave do Gemini sem expô-la no código do notebook."""
    try:
        from google.colab import userdata  # type: ignore

        chave = userdata.get("GEMINI_API_KEY")
        if chave:
            print("Chave obtida dos Secrets do Colab.")
            return chave
    except Exception:
        pass

    chave = os.environ.get("GEMINI_API_KEY")
    if chave:
        print("Chave obtida da variável de ambiente GEMINI_API_KEY.")
        return chave

    from getpass import getpass

    chave = getpass("Cole sua GEMINI_API_KEY (a digitação fica oculta): ").strip()
    if not chave:
        raise RuntimeError(
            "Nenhuma chave de API foi fornecida. Gere uma em https://aistudio.google.com/apikey"
        )
    return chave


cliente_genai = genai.Client(
    api_key=obter_api_key(),
    http_options=types.HttpOptions(
        timeout=TIMEOUT_CHAMADA_MS,
        retry_options=types.HttpRetryOptions(attempts=1),  # retry fica na seção 3.4
    ),
)

MODELOS_DISPONIVEIS: list[str] = []
for modelo in MODELOS:
    try:
        cliente_genai.models.get(model=modelo)
        MODELOS_DISPONIVEIS.append(modelo)
        print(f"  {modelo:<24} disponível")
    except errors.APIError as erro:
        print(f"  {modelo:<24} indisponível para esta chave (HTTP {erro.code})")

if not MODELOS_DISPONIVEIS:
    raise RuntimeError(
        f"Nenhum dos modelos {MODELOS} está acessível. "
        "Confira se a GEMINI_API_KEY é válida."
    )
print(f"Conexão verificada sem consumo de cota · modelos em uso: {MODELOS_DISPONIVEIS}")

## 3.4 Camada de acesso ao modelo

Toda chamada ao Gemini — geração ou embedding — passa por `ClienteLLM`. Essa camada concentra o tratamento das falhas da API observadas durante o desenvolvimento:

| Situação | Detecção | Resposta |
|---|---|---|
| Sobrecarga do modelo (503) ou erro interno (500, 502, 504) | Código HTTP | Nova tentativa após 30 s e, se necessário, após 60 s |
| Limite por minuto (429 com `retryDelay` curto) | `retryDelay` sugerido pela API | Espera o tempo sugerido e tenta de novo |
| Cota diária esgotada (429 com `retryDelay` de horas) | `retryDelay` maior que `ESPERA_MAXIMA_ACEITAVEL` | **Não** tenta de novo: lança `CotaEsgotadaError` na hora e passa a recusar novas chamadas na sessão, sem insistir na API |
| Timeout ou queda de conexão | Exceção de rede (`httpx`) | Nova tentativa com backoff, como um 503 |
| Modelo indisponível após todas as tentativas (503 ou timeout) ou com a cota do dia esgotada | Erro persistente depois do retry | Passa para o próximo modelo da lista, que vale para o restante da execução. Se a troca acontecer no meio de uma triagem, essa triagem termina em contingência: o histórico produzido por um modelo não é reaproveitado por outro |
| Erro do cliente (400, 403...) | Código HTTP | Propaga imediatamente: tentar de novo não corrige argumento inválido |

Dois pontos de projeto:

- **Poucas tentativas, com esperas longas.** Nas execuções de teste, cada tentativa repetida também contou na cota diária: uma sequência de erros 503 chegou a consumir a cota do dia com poucas triagens concluídas. Por isso são três tentativas por chamada, com esperas de 30 s e 60 s — os mesmos valores usados nas Aulas 1 e 2 —, em vez de muitas tentativas em sequência.
- **A espera é injetável** (`dormir`). Nos testes, um `dormir` falso permite simular uma sequência de erros 503 sem esperar de fato.
- **O agente depende do `ClienteLLM`, não do SDK.** Na seção de testes, um modelo simulado implementa a mesma interface (`gerar`), o que permite testar limites e recuperação de falhas de forma determinística e sem gastar cota.

In [ ]:
class CotaEsgotadaError(RuntimeError):
    """Cota do modelo esgotada, sem liberação em tempo útil (ex.: limite diário do free tier)."""


CODIGOS_TRANSITORIOS = {429, 500, 502, 503, 504}


def _retry_delay_segundos(erro: errors.APIError) -> float | None:
    """Extrai o retryDelay sugerido pela API (ex.: '19144s'), quando presente."""
    texto = json.dumps(getattr(erro, "details", None) or {}, default=str) + " " + str(erro)
    achado = re.search(r"retryDelay\W+(\d+(?:\.\d+)?)s", texto)
    return float(achado.group(1)) if achado else None


def chamar_com_retry(
    funcao: Callable[[], Any],
    *,
    descricao: str,
    tentativas: int = TENTATIVAS_POR_CHAMADA,
    espera_base: float = ESPERA_BASE_SEGUNDOS,
    dormir: Callable[[float], None] = time.sleep,
) -> Any:
    """Executa `funcao` com retry e backoff exponencial para erros transitórios da API e da rede."""
    for tentativa in range(1, tentativas + 1):
        try:
            return funcao()
        except errors.APIError as erro:
            codigo = getattr(erro, "code", None)
            if codigo not in CODIGOS_TRANSITORIOS:
                raise
            sugerido = _retry_delay_segundos(erro)
            if codigo == 429 and sugerido and sugerido > ESPERA_MAXIMA_ACEITAVEL:
                raise CotaEsgotadaError(
                    f"Cota esgotada em '{descricao}': a API pede {sugerido / 3600:.1f} h de espera."
                ) from erro
            if tentativa == tentativas:
                raise
            espera = (sugerido or espera_base * 2 ** (tentativa - 1)) + random.uniform(0, 1)
            print(f"  [{descricao}] API respondeu {codigo} — tentativa {tentativa}/{tentativas}; "
                  f"nova tentativa em {espera:.0f}s")
            dormir(espera)
        except (httpx.TimeoutException, httpx.TransportError) as erro:
            # Timeout ou queda de conexão: a resposta não chegou. É transitório como um 503.
            if tentativa == tentativas:
                raise
            espera = espera_base * 2 ** (tentativa - 1) + random.uniform(0, 1)
            print(f"  [{descricao}] {type(erro).__name__} — tentativa {tentativa}/{tentativas}; "
                  f"nova tentativa em {espera:.0f}s")
            dormir(espera)
    raise RuntimeError(f"[{descricao}] falha após {tentativas} tentativas.")


class ClienteLLM:
    """Ponto único de acesso ao Gemini: retry, modelos de reserva, cota e contagem de chamadas."""

    def __init__(self, cliente, modelos: str | list[str], modelo_embedding: str,
                 dormir: Callable[[float], None] = time.sleep):
        self._cliente = cliente
        self.modelos = [modelos] if isinstance(modelos, str) else list(modelos)
        self._indice_modelo = 0
        self.modelo_embedding = modelo_embedding
        self._dormir = dormir
        self.chamadas_geracao = 0
        self.chamadas_embedding = 0
        self.trocas_de_modelo: list[str] = []
        self.cota_esgotada = False          # cota esgotada em todos os modelos: não insiste na API

    @property
    def modelo(self) -> str:
        return self.modelos[self._indice_modelo]

    def _passar_para_reserva(self, motivo: str) -> bool:
        """Passa para o próximo modelo da lista. Devolve False se não houver reserva."""
        if self._indice_modelo + 1 >= len(self.modelos):
            return False
        anterior = self.modelo
        self._indice_modelo += 1
        self.trocas_de_modelo.append(f"{anterior} → {self.modelo} ({motivo})")
        print(f"  [geração] {anterior} indisponível ({motivo}); as próximas chamadas usam {self.modelo}.")
        return True

    def gerar(self, contents: list[types.Content],
              config: types.GenerateContentConfig) -> types.GenerateContentResponse:
        """Uma chamada de geração (uma rodada do agente).

        Se o modelo atual falhar depois de todas as tentativas, passa para o modelo de reserva.
        No início de uma conversa (nenhuma resposta de modelo no histórico), a mesma chamada é
        repetida no modelo de reserva; no meio dela, o erro é propagado.
        """
        if self.cota_esgotada:
            raise CotaEsgotadaError("a cota diária dos modelos configurados já se esgotou nesta sessão; "
                                    "chamada não enviada.")
        inicio_da_conversa = not any(c.role == "model" for c in contents)
        while True:
            modelo = self.modelo
            self.chamadas_geracao += 1
            try:
                return chamar_com_retry(
                    lambda: self._cliente.models.generate_content(
                        model=modelo, contents=contents, config=config
                    ),
                    descricao=f"geração · {modelo}",
                    dormir=self._dormir,
                )
            except CotaEsgotadaError:
                if not self._passar_para_reserva("cota diária esgotada"):
                    self.cota_esgotada = True
                    raise
                if not inicio_da_conversa:
                    raise
            except (errors.ServerError, httpx.TimeoutException, httpx.TransportError) as erro:
                if not self._passar_para_reserva(type(erro).__name__) or not inicio_da_conversa:
                    raise

    def embed(self, textos: list[str], tipo: str, lote: int = 100) -> list[list[float]]:
        """Embeddings em lotes. tipo: 'RETRIEVAL_DOCUMENT' ou 'RETRIEVAL_QUERY'."""
        vetores: list[list[float]] = []
        for inicio in range(0, len(textos), lote):
            parte = textos[inicio:inicio + lote]
            resposta = chamar_com_retry(
                lambda: self._cliente.models.embed_content(
                    model=self.modelo_embedding,
                    contents=parte,
                    config=types.EmbedContentConfig(
                        task_type=tipo, output_dimensionality=DIMENSAO_EMBEDDING
                    ),
                ),
                descricao="embedding",
                dormir=self._dormir,
            )
            self.chamadas_embedding += 1
            vetores.extend(list(e.values) for e in resposta.embeddings)
        return vetores

    def resumo(self) -> str:
        texto = (f"modelo: {self.modelo} · chamadas de geração: {self.chamadas_geracao} · "
                 f"requisições de embedding: {self.chamadas_embedding}")
        if self.trocas_de_modelo:
            texto += f" · trocas de modelo: {len(self.trocas_de_modelo)}"
        if self.cota_esgotada:
            texto += " · cota diária esgotada"
        return texto


llm = ClienteLLM(cliente_genai, MODELOS_DISPONIVEIS, MODELO_EMBEDDING)
print("ClienteLLM pronto —", llm.resumo())

---
# 4. Dados operacionais

As tabelas representam os sistemas da Apollo: cadastro da frota, ordens de serviço, histórico de corretivas, plano e registro de preventivas e estoque. Elas são baixadas do repositório e **validadas antes de qualquer uso** — a primeira camada de validação de entradas do sistema.

## 4.1 Download com fallback

`baixar_arquivo` tenta o download três vezes. Se o GitHub estiver indisponível, usa a cópia local de uma execução anterior, se existir. Isso também permite rodar o notebook sem internet: basta colocar os arquivos em `apollo_dados/`, na mesma estrutura do repositório.

In [ ]:
def baixar_arquivo(caminho_relativo: str, tentativas: int = 3) -> Path:
    """Baixa um arquivo do repositório; em falha, recorre à cópia local, se houver."""
    destino = PASTA_LOCAL / caminho_relativo
    destino.parent.mkdir(parents=True, exist_ok=True)
    url = REPOSITORIO_RAW + caminho_relativo
    ultimo_erro: Exception | None = None

    for tentativa in range(1, tentativas + 1):
        try:
            with urllib.request.urlopen(url, timeout=30) as resposta:
                conteudo = resposta.read()
            if not conteudo:
                raise ValueError("arquivo vazio")
            destino.write_bytes(conteudo)
            return destino
        except Exception as erro:  # rede, HTTP 404/5xx, timeout
            ultimo_erro = erro
            if tentativa < tentativas:
                time.sleep(2 * tentativa)

    if destino.exists() and destino.stat().st_size > 0:
        print(f"  aviso: download de {caminho_relativo} falhou ({ultimo_erro}); usando cópia local.")
        return destino
    raise RuntimeError(f"Não foi possível obter {caminho_relativo} ({url}): {ultimo_erro}")

## 4.2 Carga e validação

Cada tabela tem um contrato: colunas obrigatórias, colunas numéricas e chave única. Depois da carga, a integridade entre tabelas também é conferida — por exemplo, todo equipamento precisa pertencer a uma obra cadastrada. Dado inconsistente interrompe o notebook aqui, com mensagem clara, em vez de produzir uma triagem errada mais adiante.

In [ ]:
class DadosInvalidosError(ValueError):
    """Tabela fora do contrato esperado."""


CONTRATO_TABELAS: dict[str, dict[str, list[str]]] = {
    "obras": {
        "colunas": ["obra_id", "nome", "municipio_uf", "almoxarifado_id"],
        "numericas": [], "chave": ["obra_id"]},
    "equipamentos": {
        "colunas": ["equipamento_id", "tipo", "fabricante", "modelo", "ano_fabricacao", "obra_id",
                    "status_operacional", "criticidade", "horimetro_atual", "ficha_tecnica"],
        "numericas": ["ano_fabricacao", "horimetro_atual"], "chave": ["equipamento_id"]},
    "ordens_servico": {
        "colunas": ["os_id", "data_abertura", "obra_id", "equipamento_informado", "aberta_por", "relato"],
        "numericas": [], "chave": ["os_id"]},
    "historico_manutencao": {
        "colunas": ["ordem_id", "equipamento_id", "data_abertura", "data_conclusao", "tipo_manutencao",
                    "sistema", "codigo_alarme", "descricao_servico", "pecas_utilizadas", "horas_parado"],
        "numericas": [], "chave": ["ordem_id"]},
    "plano_preventiva": {
        "colunas": ["ficha_tecnica", "revisao", "intervalo_horas"],
        "numericas": ["intervalo_horas"], "chave": ["ficha_tecnica", "revisao"]},
    "registro_preventivas": {
        "colunas": ["equipamento_id", "revisao", "horimetro_execucao", "data_execucao"],
        "numericas": ["horimetro_execucao"], "chave": ["equipamento_id", "revisao"]},
    "estoque_pecas": {
        "colunas": ["codigo_peca", "descricao", "ref_fabricante", "ficha_tecnica", "almoxarifado_id",
                    "saldo", "unidade", "prazo_reposicao_dias"],
        "numericas": ["saldo", "prazo_reposicao_dias"], "chave": ["codigo_peca", "almoxarifado_id"]},
}


def carregar_tabela(nome: str) -> pd.DataFrame:
    """Baixa e valida uma tabela contra o seu contrato."""
    contrato = CONTRATO_TABELAS[nome]
    df = pd.read_csv(baixar_arquivo(f"dados/{nome}.csv"), dtype=str,
                     keep_default_na=False, encoding="utf-8")

    faltando = [c for c in contrato["colunas"] if c not in df.columns]
    if faltando:
        raise DadosInvalidosError(f"{nome}: colunas ausentes {faltando}")

    for coluna in contrato["numericas"]:
        convertida = pd.to_numeric(df[coluna], errors="coerce")
        invalidas = df.loc[convertida.isna(), coluna].tolist()
        if invalidas:
            raise DadosInvalidosError(f"{nome}.{coluna}: valores não numéricos {invalidas[:3]}")
        df[coluna] = convertida.astype(int)

    duplicadas = df[df.duplicated(subset=contrato["chave"], keep=False)]
    if not duplicadas.empty:
        raise DadosInvalidosError(f"{nome}: chave {contrato['chave']} duplicada em {len(duplicadas)} linhas")
    return df


def validar_integridade(t: dict[str, pd.DataFrame]) -> None:
    """Confere as referências entre tabelas."""
    verificacoes = [
        ("equipamentos.obra_id", t["equipamentos"]["obra_id"], t["obras"]["obra_id"]),
        ("equipamentos.ficha_tecnica", t["equipamentos"]["ficha_tecnica"], t["plano_preventiva"]["ficha_tecnica"]),
        ("historico.equipamento_id", t["historico_manutencao"]["equipamento_id"], t["equipamentos"]["equipamento_id"]),
        ("registro_preventivas.equipamento_id", t["registro_preventivas"]["equipamento_id"], t["equipamentos"]["equipamento_id"]),
        ("ordens_servico.obra_id", t["ordens_servico"]["obra_id"], t["obras"]["obra_id"]),
    ]
    for rotulo, valores, validos in verificacoes:
        orfaos = sorted(set(valores) - set(validos))
        if orfaos:
            raise DadosInvalidosError(f"{rotulo}: referências inexistentes {orfaos}")
    # ordens_servico.equipamento_informado NÃO é verificado de propósito:
    # o que o operador digita pode estar errado — tratar isso é tarefa do agente.


TABELAS = {nome: carregar_tabela(nome) for nome in CONTRATO_TABELAS}
validar_integridade(TABELAS)

print("Tabelas carregadas e validadas:")
for nome, df in TABELAS.items():
    print(f"  {nome:<22} {len(df):>3} linhas · {len(df.columns):>2} colunas")

## 4.3 Visão rápida dos dados

A frota e as ordens de serviço que serão triadas. Repare que a OS-2026-0103 informa um equipamento (`EH-15`) que não existe no cadastro, e que a OS-2026-0106 chegou sem relato — situações que o agente precisa tratar.

In [ ]:
display(TABELAS["equipamentos"][["equipamento_id", "tipo", "modelo", "obra_id",
                                   "status_operacional", "criticidade", "horimetro_atual"]])

In [ ]:
with pd.option_context("display.max_colwidth", 110):
    display(TABELAS["ordens_servico"][["os_id", "obra_id", "equipamento_informado", "relato"]])

---
# 5. Base de conhecimento técnica (Vector RAG)

## 5.1 Documentos

Sete documentos técnicos da Apollo, em PDF. O código do documento é o identificador usado nas citações do parecer.

In [ ]:
DOCUMENTOS_BASE: dict[str, str] = {
    "FTM-CAT320": "Ficha Técnica de Manutenção — Escavadeira Hidráulica CAT 320",
    "FTM-CATD9T": "Ficha Técnica de Manutenção — Trator de Esteiras CAT D9T",
    "FTM-CAT924H": "Ficha Técnica de Manutenção — Carregadeira de Rodas CAT 924H",
    "FTM-MBCAM": "Ficha Técnica de Manutenção — Caminhões Basculantes Mercedes-Benz",
    "CAT-ALM-001": "Catálogo de Alarmes de Telemetria e Ocorrências",
    "PRO-SEG-001": "Procedimento de Segurança — Bloqueio e Impedimento de Operação",
    "POL-MAN-001": "Política de Manutenção da Frota de Equipamentos",
}

ARQUIVOS_PDF = {codigo: baixar_arquivo(f"documentos/{codigo}.pdf") for codigo in DOCUMENTOS_BASE}
for codigo, caminho in ARQUIVOS_PDF.items():
    print(f"  {codigo:<12} {len(PdfReader(caminho).pages)} página(s) · {DOCUMENTOS_BASE[codigo]}")

## 5.2 Chunking por seção

Os documentos têm seções numeradas (`5 SISTEMA DE FREIOS`) e subseções (`5.3 Conduta e diagnóstico`). Cada seção ou subseção vira um chunk, o que traz três vantagens sobre o chunking por número fixo de caracteres:

- **O chunk é uma unidade de sentido.** Um procedimento não é cortado no meio.
- **A citação é precisa.** O agente cita `FTM-CAT924H § 5.3`, e o planejador encontra o trecho na hora.
- **Os metadados permitem filtrar.** Cada chunk carrega a sua `aplicacao`: o código da ficha técnica, ou `geral` para catálogo, procedimento de segurança e política.

Na limpeza, cabeçalhos e rodapés de página são descartados. Seções muito longas seriam subdivididas por linhas, mas nenhuma seção atual chega a esse limite.

In [ ]:
RE_SECAO = re.compile(r"^(\d{1,2}) ([A-ZÁÉÍÓÚÂÊÔÃÕÇ][A-ZÁÉÍÓÚÂÊÔÃÕÇ ,—\-]+)$")
RE_SUBSECAO = re.compile(r"^(\d{1,2}\.\d{1,2}) ([A-ZÁÉÍÓÚÂÊÔÃÕÇ].*)$")
RE_RUIDO = re.compile(r"^(Construtora Apollo S\.A\. · .*|Página \d+)$")
TAMANHO_MAXIMO_CHUNK = 1800


def _linhas_do_pdf(caminho: Path) -> list[str]:
    """Texto do PDF em linhas, sem cabeçalhos e rodapés de página."""
    linhas = []
    for pagina in PdfReader(caminho).pages:
        for linha in (pagina.extract_text() or "").splitlines():
            linha = linha.strip()
            if linha and not RE_RUIDO.match(linha):
                linhas.append(linha)
    return linhas


def _dividir(texto: str, limite: int) -> list[str]:
    """Divide um texto longo em partes, respeitando quebras de linha."""
    if len(texto) <= limite:
        return [texto]
    partes, atual = [], ""
    for linha in texto.split("\n"):
        if atual and len(atual) + len(linha) + 1 > limite:
            partes.append(atual)
            atual = ""
        atual = f"{atual}\n{linha}" if atual else linha
    return partes + ([atual] if atual else [])


def segmentar_documento(codigo: str, caminho: Path) -> list[dict[str, Any]]:
    """Transforma um PDF em chunks por seção, com metadados para filtro e citação."""
    secoes: list[dict[str, Any]] = []
    titulo_pai = ""
    for linha in _linhas_do_pdf(caminho):
        m_secao, m_sub = RE_SECAO.match(linha), RE_SUBSECAO.match(linha)
        if m_secao:
            titulo_pai = m_secao.group(2).capitalize()
            secoes.append({"secao": m_secao.group(1), "titulo": titulo_pai, "linhas": []})
        elif m_sub:
            secoes.append({"secao": m_sub.group(1),
                           "titulo": f"{titulo_pai} › {m_sub.group(2)}", "linhas": []})
        elif secoes:                       # descarta o preâmbulo antes da seção 1
            secoes[-1]["linhas"].append(linha)

    aplicacao = codigo if codigo.startswith("FTM-") else "geral"
    chunks = []
    for secao in secoes:
        texto = "\n".join(secao["linhas"]).strip()
        if not texto:                      # seção que só agrupa subseções
            continue
        for parte, trecho in enumerate(_dividir(texto, TAMANHO_MAXIMO_CHUNK), start=1):
            assinatura = hashlib.sha1(trecho.encode("utf-8")).hexdigest()[:8]
            chunks.append({
                "id": f"{codigo}§{secao['secao']}#{parte}-{assinatura}",
                "texto": trecho,
                "metadata": {
                    "documento": codigo,
                    "titulo_documento": DOCUMENTOS_BASE[codigo],
                    "secao": secao["secao"],
                    "titulo": secao["titulo"],
                    "referencia": f"{codigo} § {secao['secao']}",
                    "aplicacao": aplicacao,
                },
            })
    return chunks


CHUNKS = [c for codigo, caminho in ARQUIVOS_PDF.items() for c in segmentar_documento(codigo, caminho)]

resumo_chunks = pd.DataFrame([c["metadata"] | {"caracteres": len(c["texto"])} for c in CHUNKS])
print(f"{len(CHUNKS)} chunks · média de {resumo_chunks['caracteres'].mean():.0f} caracteres "
      f"· maior com {resumo_chunks['caracteres'].max()}")
display(resumo_chunks.groupby("documento").agg(chunks=("referencia", "count"),
                                                 aplicacao=("aplicacao", "first")))

In [ ]:
# Exemplo de chunk: o procedimento de conduta em falha de freio da carregadeira
exemplo = next(c for c in CHUNKS if c["metadata"]["referencia"] == "FTM-CAT924H § 5.3")
print(exemplo["metadata"]["referencia"], "—", exemplo["metadata"]["titulo"])
print("-" * 78)
print(exemplo["texto"])

## 5.3 Embeddings e índice vetorial

O índice usa ChromaDB persistente com distância de cosseno. O texto vetorizado inclui o título do documento e da seção — isso ajuda a busca a associar *"conduta e diagnóstico"* ao contexto de *"sistema de freios da carregadeira"*.

A indexação é **idempotente**: o id de cada chunk inclui um hash do conteúdo. Reexecutar a célula não gera novos embeddings; se um documento mudar no repositório, só os chunks alterados são reindexados e os obsoletos são removidos. Toda a base cabe em uma única requisição de embedding.

In [ ]:
DIRETORIO_CHROMA = "chroma_db"
NOME_COLECAO = "apollo_base_tecnica"

cliente_chroma = chromadb.PersistentClient(path=DIRETORIO_CHROMA)
colecao = cliente_chroma.get_or_create_collection(
    name=NOME_COLECAO, metadata={"hnsw:space": "cosine"}
)

ids_atuais = [c["id"] for c in CHUNKS]
obsoletos = sorted(set(colecao.get()["ids"]) - set(ids_atuais))
if obsoletos:
    colecao.delete(ids=obsoletos)

ja_indexados = set(colecao.get(ids=ids_atuais)["ids"])
novos = [c for c in CHUNKS if c["id"] not in ja_indexados]

if novos:
    vetores = llm.embed(
        [f"{c['metadata']['titulo_documento']} — {c['metadata']['titulo']}\n{c['texto']}" for c in novos],
        tipo="RETRIEVAL_DOCUMENT",
    )
    colecao.upsert(
        ids=[c["id"] for c in novos],
        embeddings=vetores,
        documents=[c["texto"] for c in novos],
        metadatas=[c["metadata"] for c in novos],
    )

print(f"Índice pronto: {colecao.count()} chunks · novos nesta execução: {len(novos)} "
      f"· removidos: {len(obsoletos)}")
print(llm.resumo())

## 5.4 Busca

`buscar_trechos` é o mecanismo de recuperação. Na seção 6 ele será exposto ao agente como a ferramenta `buscar_documentacao_tecnica`.

- **Filtro por aplicação.** Com `ficha_tecnica` informada, a busca considera só a ficha daquele equipamento e os documentos gerais.
- **Cache de consultas.** O vetor de cada pergunta é guardado em memória; repetir a mesma pergunta não gera nova requisição.

In [ ]:
@lru_cache(maxsize=256)
def _vetor_da_consulta(pergunta: str) -> tuple[float, ...]:
    return tuple(llm.embed([pergunta], tipo="RETRIEVAL_QUERY")[0])


def buscar_trechos(pergunta: str, ficha_tecnica: str | None = None, k: int = 3) -> list[dict[str, Any]]:
    """Recupera os k trechos mais próximos da pergunta, opcionalmente filtrando por ficha técnica."""
    filtro = {"aplicacao": {"$in": [ficha_tecnica, "geral"]}} if ficha_tecnica else None
    resultado = colecao.query(
        query_embeddings=[list(_vetor_da_consulta(pergunta.strip()))],
        n_results=k,
        where=filtro,
        include=["documents", "metadatas", "distances"],
    )
    return [
        {
            "referencia": meta["referencia"],
            "documento": meta["titulo_documento"],
            "titulo": meta["titulo"],
            "aplicacao": meta["aplicacao"],
            "trecho": texto,
            "distancia": round(distancia, 4),
        }
        for texto, meta, distancia in zip(
            resultado["documents"][0], resultado["metadatas"][0], resultado["distances"][0]
        )
    ]

## 5.5 Verificação da recuperação

Antes de entregar a busca ao agente, conferimos se ela encontra o trecho certo para perguntas típicas da triagem. São cinco consultas, uma requisição de embedding cada:

- **Posição do esperado:** em que lugar do top-3 aparece a seção que responde à pergunta.
- **Filtro respeitado:** quando há ficha técnica, nenhum trecho de outra ficha pode aparecer. Essa verificação é determinística e interrompe o notebook se falhar.

In [ ]:
CONSULTAS_DE_VERIFICACAO = [
    ("alarme TL-214 lança e braço lentos depois que a máquina aquece", "FTM-CAT320", "FTM-CAT320 § 5.2"),
    ("pedal de freio baixo e carregadeira demorando para parar", "FTM-CAT924H", "FTM-CAT924H § 5.2"),
    ("quando é obrigatório bloquear o equipamento", None, "PRO-SEG-001 § 3"),
    ("falha no freio de serviço obriga a bloquear o equipamento?", None, "PRO-SEG-001 § 3"),
    ("quantas ocorrências caracterizam falha recorrente", None, "POL-MAN-001 § 6"),
    ("ruído na dianteira do caminhão, quais informações a ordem de serviço precisa ter",
     "FTM-MBCAM", "FTM-MBCAM § 5.1"),
]

linhas = []
for pergunta, ficha, esperada in CONSULTAS_DE_VERIFICACAO:
    trechos = buscar_trechos(pergunta, ficha_tecnica=ficha)
    referencias = [t["referencia"] for t in trechos]
    if ficha:
        fora_do_filtro = [t["referencia"] for t in trechos if t["aplicacao"] not in (ficha, "geral")]
        assert not fora_do_filtro, f"Filtro violado para {ficha}: {fora_do_filtro}"
    linhas.append({
        "pergunta": pergunta[:60] + ("…" if len(pergunta) > 60 else ""),
        "filtro": ficha or "—",
        "esperada": esperada,
        "posição": referencias.index(esperada) + 1 if esperada in referencias else "fora do top-3",
        "top-3 recuperado": " | ".join(referencias),
    })

with pd.option_context("display.max_colwidth", 80):
    display(pd.DataFrame(linhas))

acertos = sum(isinstance(l["posição"], int) for l in linhas)
print(f"Esperada no top-3 em {acertos}/{len(linhas)} consultas · filtro por aplicação respeitado em todas.")
print(llm.resumo())

**Comentando o resultado.** As duas consultas sobre bloqueio procuram a mesma seção, o PRO-SEG-001 § 3, que lista as situações de bloqueio obrigatório. Na execução de referência, a pergunta genérica trouxe as seções 7, 5 e 1 do mesmo procedimento (responsabilidades, procedimento e objetivo): o documento certo, mas não a seção que responde. A pergunta específica, formulada como o agente pergunta durante uma triagem, trouxe a seção 3 em primeiro lugar. Várias seções do procedimento falam de "bloquear", e o sintoma concreto é o que as diferencia. Por isso a descrição da ferramenta de documentação (seção 6) orienta o modelo a fazer perguntas específicas, citando o sintoma, o sistema ou o código de alarme, uma por assunto.

---
# 6. Ferramentas

As ferramentas são a interface entre o agente e o ambiente. O modelo **nunca** acessa tabelas ou o índice vetorial diretamente: ele vê apenas o **contrato** de cada ferramenta (nome, descrição e schema dos argumentos) e pede uma chamada. Quem valida, executa e responde é o runtime.

Esta seção segue as três decisões de *tool engineering* da Aula 3:

| Decisão | Como foi aplicada |
|---|---|
| **Granularidade** | Uma ferramenta por pergunta da triagem (seção 1.2). Nem fina demais — o histórico já devolve as contagens por sistema, sem exigir uma chamada por sistema —, nem grossa demais — não existe uma ferramenta "triar OS" que esconda as decisões |
| **Camada adaptadora** | Normaliza identificadores (`eh 07` → `EH-07`), aplica a data de referência, calcula situação de preventiva e saldo por almoxarifado |
| **Formatação do retorno** | Devolve só o que o agente precisa para decidir: a seção recuperada, sem distâncias vetoriais nem colunas internas |

## 6.1 Status padronizado

Toda ferramenta responde com um dos quatro status abaixo. A distinção entre `not_found` e `error` é central para o agente: **ausência de informação é um fato** que ele deve declarar na triagem; **falha** é algo que ele deve contornar.

| Status | Significado | O que o agente deve fazer |
|---|---|---|
| `success` | Encontrou o que foi pedido | Usar como evidência |
| `not_found` | A consulta é válida, mas não há registro | Declarar a ausência; não inventar o dado |
| `invalid_arguments` | Os argumentos violam o schema | Corrigir os argumentos e tentar de novo |
| `error` | Falha interna | Seguir com outras fontes e registrar pendência |

In [ ]:
STATUS_VALIDOS = ("success", "not_found", "invalid_arguments", "error")

SISTEMAS = ["Hidráulico", "Motor", "Arrefecimento", "Elétrico", "Trem de força", "Freios",
            "Direção", "Suspensão", "Material rodante", "Estrutura e implementos"]
TIPOS_EQUIPAMENTO = sorted(TABELAS["equipamentos"]["tipo"].unique())
OBRAS = sorted(TABELAS["obras"]["obra_id"])
FICHAS_TECNICAS = [codigo for codigo in DOCUMENTOS_BASE if codigo.startswith("FTM-")]

JANELA_RECORRENCIA_DIAS = 90      # POL-MAN-001 § 6
TOLERANCIA_PREVENTIVA = 0.10      # POL-MAN-001 § 7
LIMITE_TRECHO = 1500              # cobre a maior seção da base: o trecho chega inteiro ao modelo
LIMITE_RETORNO = 8000             # tamanho máximo de um retorno serializado


def _sucesso(**dados) -> dict[str, Any]:
    return {"status": "success", **dados}


def _nao_encontrado(mensagem: str, **dados) -> dict[str, Any]:
    return {"status": "not_found", "mensagem": mensagem, **dados}


def _argumento_invalido(mensagem: str) -> dict[str, Any]:
    return {"status": "invalid_arguments", "mensagem": mensagem}


def _normalizar_id_equipamento(valor: str) -> str:
    """'eh 07', 'EH_07' e 'EH07' viram 'EH-07'."""
    texto = re.sub(r"[\s_]+", "-", valor.strip().upper())
    if re.fullmatch(r"[A-Z]{2}\d{2}", texto):
        texto = f"{texto[:2]}-{texto[2:]}"
    return texto


def _encurtar(texto: str, limite: int = LIMITE_TRECHO) -> str:
    if len(texto) <= limite:
        return texto
    return texto[:limite].rsplit(" ", 1)[0] + " […]"


def _localizar_equipamento(equipamento_id: str) -> pd.Series | None:
    eq = TABELAS["equipamentos"]
    linha = eq[eq["equipamento_id"] == _normalizar_id_equipamento(equipamento_id)]
    return None if linha.empty else linha.iloc[0]


def _resumo_equipamento(linha: pd.Series) -> dict[str, Any]:
    return {
        "equipamento_id": linha["equipamento_id"],
        "tipo": linha["tipo"],
        "modelo": linha["modelo"],
        "obra_id": linha["obra_id"],
        "status_operacional": linha["status_operacional"],
        "criticidade": linha["criticidade"],
        "horimetro_atual": int(linha["horimetro_atual"]),
        "ficha_tecnica": linha["ficha_tecnica"],
    }


print(f"{len(SISTEMAS)} sistemas · {len(TIPOS_EQUIPAMENTO)} tipos de equipamento · "
      f"{len(OBRAS)} obras · {len(FICHAS_TECNICAS)} fichas técnicas")

## 6.2 Implementação

Cada função abaixo é a implementação de uma ferramenta. Todas leem as tabelas da seção 4 ou o índice da seção 5 e devolvem um dicionário com `status`. Os cálculos — horas desde a última revisão, contagens de recorrência, saldo por almoxarifado — são feitos aqui, em Python, e não pelo modelo.

In [ ]:
def buscar_documentacao_tecnica(pergunta: str, ficha_tecnica: str | None = None) -> dict[str, Any]:
    """Recupera trechos da documentação técnica (Vector RAG)."""
    trechos = buscar_trechos(pergunta, ficha_tecnica=ficha_tecnica, k=3)
    if not trechos:
        return _nao_encontrado("Nenhum trecho da documentação corresponde à pergunta.")
    return _sucesso(
        pergunta=pergunta,
        filtro=ficha_tecnica or "todos os documentos",
        evidencias=[
            {"referencia": t["referencia"], "documento": t["documento"],
             "secao": t["titulo"], "trecho": _encurtar(t["trecho"])}
            for t in trechos
        ],
    )


def consultar_equipamento(equipamento_id: str | None = None, obra_id: str | None = None,
                          tipo: str | None = None) -> dict[str, Any]:
    """Localiza um equipamento pelo prefixo ou lista candidatos por obra e tipo."""
    if not (equipamento_id or obra_id or tipo):
        return _argumento_invalido("Informe equipamento_id, ou obra_id e/ou tipo para listar candidatos.")

    eq = TABELAS["equipamentos"]
    candidatos = eq
    if obra_id:
        candidatos = candidatos[candidatos["obra_id"] == obra_id]
    if tipo:
        candidatos = candidatos[candidatos["tipo"] == tipo]
    lista_candidatos = [_resumo_equipamento(l) for _, l in candidatos.iterrows()] if (obra_id or tipo) else []

    if equipamento_id:
        linha = _localizar_equipamento(equipamento_id)
        if linha is not None:
            return _sucesso(equipamento=_resumo_equipamento(linha))
        return _nao_encontrado(
            f"O equipamento {_normalizar_id_equipamento(equipamento_id)} não existe no cadastro.",
            candidatos=lista_candidatos,
            observacao="Candidatos apenas atendem aos filtros informados; não indicam qual é o "
                       "equipamento da ordem de serviço.",
        )

    if not lista_candidatos:
        return _nao_encontrado("Nenhum equipamento atende aos filtros informados.")
    return _sucesso(quantidade=len(lista_candidatos), candidatos=lista_candidatos)


def consultar_historico_manutencao(equipamento_id: str, sistema: str | None = None,
                                   janela_dias: int = 180) -> dict[str, Any]:
    """Histórico de manutenções corretivas do equipamento e contagem por sistema em 90 dias."""
    linha = _localizar_equipamento(equipamento_id)
    if linha is None:
        return _nao_encontrado(f"O equipamento {equipamento_id} não existe no cadastro.")
    eq_id = linha["equipamento_id"]

    hist = TABELAS["historico_manutencao"]
    hist = hist[hist["equipamento_id"] == eq_id].copy()
    hist["data"] = pd.to_datetime(hist["data_abertura"])
    referencia = pd.Timestamp(DATA_REFERENCIA)

    recentes = hist[hist["data"] >= referencia - pd.Timedelta(days=JANELA_RECORRENCIA_DIAS)]
    na_janela = hist[hist["data"] >= referencia - pd.Timedelta(days=int(janela_dias))]
    if sistema:
        na_janela = na_janela[na_janela["sistema"] == sistema]

    ocorrencias = [
        {
            "ordem_id": l["ordem_id"],
            "data_abertura": l["data_abertura"],
            "sistema": l["sistema"],
            "codigo_alarme": l["codigo_alarme"] or None,
            "servico_executado": l["descricao_servico"],
            "pecas_utilizadas": [p for p in l["pecas_utilizadas"].split(";") if p],
            "horas_parado": int(l["horas_parado"]) if l["horas_parado"] else "em andamento",
        }
        for _, l in na_janela.sort_values("data", ascending=False).iterrows()
    ]
    return _sucesso(
        equipamento_id=eq_id,
        janela_dias=int(janela_dias),
        filtro_sistema=sistema or "todos",
        total_ocorrencias=len(ocorrencias),
        ocorrencias=ocorrencias,
        corretivas_por_sistema_ultimos_90_dias=recentes.groupby("sistema").size().astype(int).to_dict(),
        observacao="Contagens e lista NÃO incluem a ordem de serviço que está em triagem.",
    )


def consultar_plano_preventiva(equipamento_id: str) -> dict[str, Any]:
    """Situação de cada revisão preventiva do equipamento, pelo horímetro atual."""
    linha = _localizar_equipamento(equipamento_id)
    if linha is None:
        return _nao_encontrado(f"O equipamento {equipamento_id} não existe no cadastro.")
    eq_id, horimetro = linha["equipamento_id"], int(linha["horimetro_atual"])

    plano = TABELAS["plano_preventiva"]
    plano = plano[plano["ficha_tecnica"] == linha["ficha_tecnica"]]
    registro = TABELAS["registro_preventivas"]
    registro = registro[registro["equipamento_id"] == eq_id].set_index("revisao")

    revisoes = []
    for _, rev in plano.iterrows():
        intervalo = int(rev["intervalo_horas"])
        if rev["revisao"] in registro.index:
            execucao = registro.loc[rev["revisao"]]
            horas_desde = horimetro - int(execucao["horimetro_execucao"])
            fracao = horas_desde / intervalo
            situacao = ("vencida" if fracao > 1 + TOLERANCIA_PREVENTIVA
                        else "proxima" if fracao >= 0.9 else "em_dia")
            revisoes.append({
                "revisao": rev["revisao"], "intervalo_horas": intervalo,
                "ultima_execucao_horimetro": int(execucao["horimetro_execucao"]),
                "ultima_execucao_data": execucao["data_execucao"],
                "horas_desde_ultima": horas_desde,
                "percentual_do_intervalo": round(100 * fracao),
                "situacao": situacao,
            })
        else:
            situacao = "nao_atingida" if horimetro < intervalo else "vencida"
            revisoes.append({"revisao": rev["revisao"], "intervalo_horas": intervalo,
                             "ultima_execucao_horimetro": None, "situacao": situacao})

    return _sucesso(
        equipamento_id=eq_id,
        ficha_tecnica=linha["ficha_tecnica"],
        horimetro_atual=horimetro,
        revisoes=revisoes,
        revisoes_vencidas=[r["revisao"] for r in revisoes if r["situacao"] == "vencida"],
        revisoes_proximas=[r["revisao"] for r in revisoes if r["situacao"] == "proxima"],
        criterio="vencida acima de 110% do intervalo; próxima a partir de 90%",
    )


def consultar_estoque_pecas(codigos_peca: list[str], obra_id: str | None = None) -> dict[str, Any]:
    """Saldo das peças por almoxarifado, destacando o almoxarifado da obra."""
    estoque = TABELAS["estoque_pecas"]
    obras = TABELAS["obras"].set_index("obra_id")
    almox_obra = obras.loc[obra_id, "almoxarifado_id"] if obra_id else None

    pecas, nao_encontrados = [], []
    for codigo in dict.fromkeys(c.strip().upper() for c in codigos_peca):
        linhas = estoque[estoque["codigo_peca"] == codigo]
        if linhas.empty:
            nao_encontrados.append(codigo)
            continue
        saldo_total = int(linhas["saldo"].sum())
        saldo_obra = int(linhas.loc[linhas["almoxarifado_id"] == almox_obra, "saldo"].sum()) if almox_obra else None
        if almox_obra and saldo_obra > 0:
            situacao = "disponivel_na_obra"
        elif saldo_total > 0:
            situacao = "disponivel_em_outro_almoxarifado" if almox_obra else "disponivel"
        else:
            situacao = "sem_estoque"
        pecas.append({
            "codigo_peca": codigo,
            "descricao": linhas.iloc[0]["descricao"],
            "ref_fabricante": linhas.iloc[0]["ref_fabricante"] or None,
            "situacao": situacao,
            "saldo_total": saldo_total,
            "saldo_no_almoxarifado_da_obra": saldo_obra,
            "almoxarifados_com_saldo": [
                {"almoxarifado_id": l["almoxarifado_id"], "saldo": int(l["saldo"])}
                for _, l in linhas[linhas["saldo"] > 0].iterrows()
            ],
            "prazo_reposicao_dias": int(linhas.iloc[0]["prazo_reposicao_dias"]),
        })

    if not pecas:
        return _nao_encontrado("Nenhum dos códigos informados existe no estoque.",
                               codigos_nao_encontrados=nao_encontrados)
    return _sucesso(obra_id=obra_id, almoxarifado_da_obra=almox_obra, pecas=pecas,
                    codigos_nao_encontrados=nao_encontrados)


def consultar_frota_disponivel(tipo: str, obra_id: str) -> dict[str, Any]:
    """Equipamentos do mesmo tipo com status Disponível, em qualquer obra."""
    eq = TABELAS["equipamentos"]
    disponiveis = eq[(eq["tipo"] == tipo) & (eq["status_operacional"] == "Disponível")].copy()
    disponiveis["na_mesma_obra"] = disponiveis["obra_id"] == obra_id
    disponiveis = disponiveis.sort_values("na_mesma_obra", ascending=False)
    lista = [
        {"equipamento_id": l["equipamento_id"], "modelo": l["modelo"], "obra_id": l["obra_id"],
         "na_mesma_obra": bool(l["na_mesma_obra"]), "horimetro_atual": int(l["horimetro_atual"])}
        for _, l in disponiveis.iterrows()
    ]
    return _sucesso(
        tipo=tipo, obra_de_referencia=obra_id, quantidade=len(lista), equipamentos=lista,
        mensagem=None if lista else "Nenhum equipamento deste tipo está disponível em nenhuma obra.",
    )


print("Seis ferramentas implementadas.")

## 6.3 Contratos e registro

O modelo recebe a definição de cada ferramenta. Por isso, além de explicar o que ela faz, a descrição orienta em quais situações deve ser usada e quais informações precisam ser passadas. A descrição da ferramenta de documentação incorpora o achado da seção 5.5: perguntas específicas, uma por assunto.

O registro `FERRAMENTAS` liga cada nome à sua implementação, ao schema dos argumentos e aos campos obrigatórios do retorno — é a integração direta, sem MCP.

In [ ]:
@dataclass(frozen=True)
class Ferramenta:
    nome: str
    descricao: str
    parametros: dict[str, Any]
    funcao: Callable[..., dict[str, Any]]
    campos_sucesso: tuple[str, ...]


ID_EQUIPAMENTO = {"type": "string", "minLength": 2, "maxLength": 12,
                  "description": "Prefixo do equipamento na frota, ex.: EH-07, CR-04, CB-12."}

FERRAMENTAS: dict[str, Ferramenta] = {f.nome: f for f in [
    Ferramenta(
        nome="buscar_documentacao_tecnica",
        descricao=(
            "Busca na documentação técnica da Construtora Apollo: fichas técnicas de manutenção "
            "por modelo, catálogo de alarmes TL, procedimento de segurança PRO-SEG-001 e política "
            "de manutenção POL-MAN-001. Use para REGRAS e CONHECIMENTO: significado de alarmes, "
            "sequência de diagnóstico, peças associadas a um sistema, critérios de bloqueio, "
            "matriz de prioridade, regra de falha recorrente e devolução de OS. "
            "Faça UMA pergunta por assunto, citando o sintoma, o sistema ou o código de alarme — ex.: "
            "'alarme TL-331 pressão baixa do óleo do freio'. NUNCA combine assuntos (ex.: prioridade + "
            "recorrência + bloqueio) numa só pergunta: cada busca devolve só 3 trechos. Para vários "
            "assuntos, faça várias chamadas na mesma rodada. Cada trecho traz a seção inteira, inclusive "
            "a linha 'Peças associadas' com os códigos APL. Informe ficha_tecnica quando o equipamento "
            "for conhecido. "
            "NÃO use para fatos operacionais (histórico, horímetro, estoque, frota)."
        ),
        parametros={
            "type": "object",
            "properties": {
                "pergunta": {"type": "string", "minLength": 10, "maxLength": 300,
                             "description": "Pergunta específica sobre um único assunto."},
                "ficha_tecnica": {"type": "string", "enum": FICHAS_TECNICAS,
                                  "description": "Ficha técnica do equipamento (campo ficha_tecnica "
                                                 "do cadastro). Omita para buscar só documentos gerais e todas as fichas."},
            },
            "required": ["pergunta"],
            "additionalProperties": False,
        },
        funcao=buscar_documentacao_tecnica,
        campos_sucesso=("evidencias",),
    ),
    Ferramenta(
        nome="consultar_equipamento",
        descricao=(
            "Consulta o cadastro da frota. Com equipamento_id, confirma se o equipamento existe e "
            "devolve tipo, modelo, obra, status, criticidade, horímetro e ficha técnica. Sem "
            "equipamento_id, lista os candidatos de uma obra e/ou tipo. Use quando o equipamento da "
            "OS não estiver identificado no contexto. Candidatos NÃO autorizam escolher um "
            "equipamento por aproximação (POL-MAN-001)."
        ),
        parametros={
            "type": "object",
            "properties": {
                "equipamento_id": ID_EQUIPAMENTO,
                "obra_id": {"type": "string", "enum": OBRAS},
                "tipo": {"type": "string", "enum": TIPOS_EQUIPAMENTO},
            },
            "additionalProperties": False,
        },
        funcao=consultar_equipamento,
        campos_sucesso=(),
    ),
    Ferramenta(
        nome="consultar_historico_manutencao",
        descricao=(
            "Histórico de manutenções CORRETIVAS do equipamento: ordens anteriores, sistema, alarme, "
            "serviço executado e peças trocadas, além da contagem de corretivas por sistema nos "
            "últimos 90 dias. Use para verificar recorrência e o que já foi tentado. A ordem em "
            "triagem não está incluída nas contagens."
        ),
        parametros={
            "type": "object",
            "properties": {
                "equipamento_id": ID_EQUIPAMENTO,
                "sistema": {"type": "string", "enum": SISTEMAS,
                            "description": "Filtra a lista de ocorrências por sistema."},
                "janela_dias": {"type": "integer", "minimum": 1, "maximum": 365,
                                "description": "Janela da lista de ocorrências. Padrão: 180."},
            },
            "required": ["equipamento_id"],
            "additionalProperties": False,
        },
        funcao=consultar_historico_manutencao,
        campos_sucesso=("ocorrencias", "corretivas_por_sistema_ultimos_90_dias"),
    ),
    Ferramenta(
        nome="consultar_plano_preventiva",
        descricao=(
            "Situação das revisões preventivas do equipamento pelo horímetro atual: em_dia, "
            "proxima ou vencida, com horas desde a última execução. Use quando o sintoma puder "
            "estar ligado a manutenção atrasada (perda de potência, filtros, aquecimento) e para "
            "informar a situação da preventiva no parecer."
        ),
        parametros={
            "type": "object",
            "properties": {"equipamento_id": ID_EQUIPAMENTO},
            "required": ["equipamento_id"],
            "additionalProperties": False,
        },
        funcao=consultar_plano_preventiva,
        campos_sucesso=("revisoes", "revisoes_vencidas"),
    ),
    Ferramenta(
        nome="consultar_estoque_pecas",
        descricao=(
            "Saldo de peças por almoxarifado, destacando o almoxarifado da obra, e prazo de "
            "reposição. Os códigos APL das peças associadas a cada sistema estão nas fichas "
            "técnicas: consulte a documentação antes para saber quais códigos verificar."
        ),
        parametros={
            "type": "object",
            "properties": {
                "codigos_peca": {"type": "array", "minItems": 1, "maxItems": 8,
                                 "items": {"type": "string", "pattern": "^APL-[0-9]{5}$"},
                                 "description": "Códigos internos no formato APL-00000."},
                "obra_id": {"type": "string", "enum": OBRAS,
                            "description": "Obra do equipamento, para destacar o almoxarifado local."},
            },
            "required": ["codigos_peca"],
            "additionalProperties": False,
        },
        funcao=consultar_estoque_pecas,
        campos_sucesso=("pecas",),
    ),
    Ferramenta(
        nome="consultar_frota_disponivel",
        descricao=(
            "Lista equipamentos do mesmo tipo com status Disponível em qualquer obra, para avaliar "
            "substituto. Use quando o equipamento puder ficar parado e a criticidade ou a "
            "prioridade dependerem da existência de substituto."
        ),
        parametros={
            "type": "object",
            "properties": {
                "tipo": {"type": "string", "enum": TIPOS_EQUIPAMENTO},
                "obra_id": {"type": "string", "enum": OBRAS},
            },
            "required": ["tipo", "obra_id"],
            "additionalProperties": False,
        },
        funcao=consultar_frota_disponivel,
        campos_sucesso=("equipamentos",),
    ),
]}

VALIDADORES = {nome: Draft202012Validator(f.parametros) for nome, f in FERRAMENTAS.items()}
for f in FERRAMENTAS.values():
    Draft202012Validator.check_schema(f.parametros)

print("Ferramentas registradas:")
for f in FERRAMENTAS.values():
    print(f"  {f.nome:<34} argumentos: {', '.join(f.parametros['properties'])}")

## 6.4 Execução controlada

A função `executar_ferramenta` valida a solicitação do modelo antes de executar a ferramenta. Ela aplica, nesta ordem:

1. **Ferramenta existe?** Nome desconhecido gera `error`, sem executar nada.
2. **Argumentos válidos?** Validação contra o schema; violação gera `invalid_arguments` com a mensagem exata do que corrigir.
3. **Execução isolada.** Exceção dentro da ferramenta vira `error` com mensagem genérica; o detalhe técnico fica no log do runtime, não no contexto do modelo.
4. **Retorno dentro do contrato?** Status válido, campos obrigatórios presentes, serializável e dentro do tamanho máximo. Se o retorno não estiver no formato esperado, ele não é enviado ao modelo e a execução retorna `error`.

In [ ]:
LOG_RUNTIME: list[dict[str, Any]] = []      # detalhes técnicos que NÃO vão para o modelo


def _formatar_erros_schema(validador: Draft202012Validator, argumentos: dict[str, Any]) -> list[str]:
    mensagens = []
    for erro in sorted(validador.iter_errors(argumentos), key=lambda e: list(e.path))[:3]:
        campo = ".".join(str(p) for p in erro.path) or "argumentos"
        mensagens.append(f"{campo}: {erro.message}")
    return mensagens


def _problema_no_retorno(ferramenta: Ferramenta, resultado: Any) -> str | None:
    if not isinstance(resultado, dict):
        return f"retorno do tipo {type(resultado).__name__}, esperado dict"
    if resultado.get("status") not in STATUS_VALIDOS:
        return f"status inválido: {resultado.get('status')!r}"
    if resultado["status"] == "success":
        faltando = [c for c in ferramenta.campos_sucesso if c not in resultado]
        if faltando:
            return f"campos obrigatórios ausentes: {faltando}"
    try:
        tamanho = len(json.dumps(resultado, ensure_ascii=False))
    except (TypeError, ValueError) as erro:
        return f"retorno não serializável: {erro}"
    if tamanho > LIMITE_RETORNO:
        return f"retorno com {tamanho} caracteres excede o limite de {LIMITE_RETORNO}"
    return None


def executar_ferramenta(nome: str, argumentos: dict[str, Any] | None,
                        registro: dict[str, Ferramenta] | None = None) -> dict[str, Any]:
    """Valida, executa e confere o retorno de uma ferramenta. Nunca lança exceção."""
    registro = registro if registro is not None else FERRAMENTAS
    ferramenta = registro.get(nome)
    if ferramenta is None:
        return {"status": "error",
                "mensagem": f"A ferramenta '{nome}' não existe. Disponíveis: {', '.join(registro)}."}

    argumentos = dict(argumentos or {})
    validador = VALIDADORES.get(nome) or Draft202012Validator(ferramenta.parametros)
    erros = _formatar_erros_schema(validador, argumentos)
    if erros:
        return _argumento_invalido("Argumentos inválidos — " + "; ".join(erros))

    try:
        resultado = ferramenta.funcao(**argumentos)
    except Exception as erro:
        LOG_RUNTIME.append({"ferramenta": nome, "argumentos": argumentos,
                            "erro": f"{type(erro).__name__}: {erro}"})
        return {"status": "error",
                "mensagem": "Falha interna ao executar a ferramenta. Prossiga com as demais fontes "
                            "e registre esta verificação como pendência."}

    problema = _problema_no_retorno(ferramenta, resultado)
    if problema:
        LOG_RUNTIME.append({"ferramenta": nome, "argumentos": argumentos,
                            "erro": f"retorno fora do contrato: {problema}"})
        return {"status": "error",
                "mensagem": "A ferramenta devolveu um resultado fora do contrato, que foi descartado. "
                            "Registre esta verificação como pendência."}
    return resultado


print("Portão de execução pronto.")

## 6.5 Declarações para o Gemini

Os contratos viram `FunctionDeclaration`s. A palavra-chave `additionalProperties` é removida da versão enviada ao modelo, por compatibilidade com o subconjunto de JSON Schema aceito em declarações de função; a validação completa, com ela, continua no runtime (seção 6.4). A ferramenta terminal `emitir_parecer` é declarada na seção 7, junto com o schema do parecer.

In [ ]:
def _schema_para_gemini(schema: Any) -> Any:
    """Cópia do schema sem palavras-chave usadas só na validação local."""
    if isinstance(schema, dict):
        return {k: _schema_para_gemini(v) for k, v in schema.items() if k != "additionalProperties"}
    if isinstance(schema, list):
        return [_schema_para_gemini(v) for v in schema]
    return schema


DECLARACOES_CONSULTA = [
    types.FunctionDeclaration(
        name=f.nome,
        description=f.descricao,
        parameters_json_schema=_schema_para_gemini(f.parametros),
    )
    for f in FERRAMENTAS.values()
]
print(f"{len(DECLARACOES_CONSULTA)} declarações de consulta prontas para o modelo.")

## 6.6 Testes isolados das ferramentas

Antes de entregar as ferramentas ao agente, cada uma é testada diretamente, sem o modelo, pelo portão `executar_ferramenta`. Os casos cobrem o caminho feliz, argumentos inválidos, ausência de informação e os fatos plantados nos dados para os cenários de teste. Todos os testes são determinísticos; apenas os dois primeiros fazem requisições de embedding.

In [ ]:
CASOS_FERRAMENTAS = [
    # (descrição, ferramenta, argumentos, status esperado, verificação adicional)
    ("documentação filtrada pela ficha", "buscar_documentacao_tecnica",
     {"pergunta": "alarme TL-331 pressão baixa do óleo do freio", "ficha_tecnica": "FTM-CAT924H"}, "success",
     lambda r: all(e["referencia"].split(" §")[0] in ("FTM-CAT924H", "CAT-ALM-001", "PRO-SEG-001", "POL-MAN-001")
                   for e in r["evidencias"])),
    ("trecho chega inteiro, com os códigos de peça", "buscar_documentacao_tecnica",
     {"pergunta": "alarme TL-214 lança e braço lentos depois de aquecer", "ficha_tecnica": "FTM-CAT320"}, "success",
     lambda r: any("APL-10021" in e["trecho"] and not e["trecho"].endswith("[…]") for e in r["evidencias"])),
    ("pergunta curta demais", "buscar_documentacao_tecnica", {"pergunta": "freio"}, "invalid_arguments", None),
    ("ficha técnica inexistente", "buscar_documentacao_tecnica",
     {"pergunta": "procedimento de bloqueio do equipamento", "ficha_tecnica": "FTM-XYZ"}, "invalid_arguments", None),
    ("equipamento existente", "consultar_equipamento", {"equipamento_id": "EH-07"}, "success",
     lambda r: r["equipamento"]["ficha_tecnica"] == "FTM-CAT320"),
    ("identificador digitado fora do padrão", "consultar_equipamento", {"equipamento_id": "eh 07"}, "success",
     lambda r: r["equipamento"]["equipamento_id"] == "EH-07"),
    ("equipamento inexistente com candidatos", "consultar_equipamento",
     {"equipamento_id": "EH-15", "obra_id": "OB-01", "tipo": "Escavadeira hidráulica"}, "not_found",
     lambda r: sorted(c["equipamento_id"] for c in r["candidatos"]) == ["EH-07", "EH-09"]),
    ("consulta sem nenhum argumento", "consultar_equipamento", {}, "invalid_arguments", None),
    ("argumento não previsto no contrato", "consultar_equipamento",
     {"equipamento_id": "EH-07", "placa": "ABC1234"}, "invalid_arguments", None),
    ("recorrência hidráulica da EH-07", "consultar_historico_manutencao",
     {"equipamento_id": "EH-07", "sistema": "Hidráulico"}, "success",
     lambda r: r["total_ocorrencias"] == 2 and r["corretivas_por_sistema_ultimos_90_dias"].get("Hidráulico") == 2),
    ("histórico de equipamento inexistente", "consultar_historico_manutencao",
     {"equipamento_id": "EH-99"}, "not_found", None),
    ("janela fora do limite", "consultar_historico_manutencao",
     {"equipamento_id": "EH-07", "janela_dias": 0}, "invalid_arguments", None),
    ("preventiva vencida do TE-02", "consultar_plano_preventiva", {"equipamento_id": "TE-02"}, "success",
     lambda r: "R500" in r["revisoes_vencidas"]),
    ("preventiva em dia da CR-04", "consultar_plano_preventiva", {"equipamento_id": "CR-04"}, "success",
     lambda r: r["revisoes_vencidas"] == []),
    ("peças de freio da CR-04", "consultar_estoque_pecas",
     {"codigos_peca": ["APL-30021", "APL-30022"], "obra_id": "OB-02"}, "success",
     lambda r: [p["situacao"] for p in r["pecas"]] == ["sem_estoque", "disponivel_em_outro_almoxarifado"]),
    ("código de peça fora do padrão", "consultar_estoque_pecas", {"codigos_peca": ["30021"]}, "invalid_arguments", None),
    ("código de peça inexistente", "consultar_estoque_pecas", {"codigos_peca": ["APL-99999"]}, "not_found", None),
    ("substituto para carregadeira", "consultar_frota_disponivel",
     {"tipo": "Carregadeira de rodas", "obra_id": "OB-02"}, "success",
     lambda r: [e["equipamento_id"] for e in r["equipamentos"]] == ["CR-06"]),
    ("ferramenta inexistente", "consultar_orcamento", {"centro_custo": "X"}, "error", None),
]

resultados_ferramentas = []
for descricao, nome, argumentos, esperado, verificacao in CASOS_FERRAMENTAS:
    retorno = executar_ferramenta(nome, argumentos)
    ok_status = retorno["status"] == esperado
    ok_fato = verificacao(retorno) if (verificacao and ok_status) else ok_status
    resultados_ferramentas.append({
        "caso": descricao, "ferramenta": nome, "esperado": esperado,
        "obtido": retorno["status"], "resultado": "OK" if (ok_status and ok_fato) else "FALHOU",
        "mensagem": (retorno.get("mensagem") or "")[:90],
    })

with pd.option_context("display.max_colwidth", 90):
    display(pd.DataFrame(resultados_ferramentas))

falhas = [r for r in resultados_ferramentas if r["resultado"] == "FALHOU"]
assert not falhas, f"{len(falhas)} caso(s) falharam: {[f['caso'] for f in falhas]}"
print(f"{len(resultados_ferramentas)} casos aprovados · {llm.resumo()}")

**Comentando o resultado.** O segundo caso é um teste de regressão. Na primeira execução do agente com o modelo real, os trechos eram cortados em 700 caracteres, e a linha "Peças associadas" da FTM-CAT320 § 5.2 ficava de fora. Sem os códigos APL, o modelo fez três buscas tentando encontrá-los. O limite passou a cobrir a maior seção da base, e esse caso confere que o trecho chega inteiro.

In [ ]:
# Exemplo de retorno completo: o que o modelo recebe ao consultar a preventiva do TE-02
print(json.dumps(executar_ferramenta("consultar_plano_preventiva", {"equipamento_id": "TE-02"}),
                 ensure_ascii=False, indent=2))

---
# 7. O agente de triagem

O agente segue as etapas de perceber, decidir e agir. Na etapa de decisão, o modelo pode consultar as ferramentas disponíveis antes de gerar o parecer da ordem de serviço. Essa etapa é um laço inspirado no padrão ReAct, implementado com Tool Calling estruturado:

```
perceber(OS)                                  [código]
   │  OS inválida? ──► rejeitada, sem chamar o modelo
   ▼
decidir(): laço                               [modelo + runtime]
   ┌─► modelo pede ferramentas ──► portão (6.4) ──► observações ─┐
   └──────────────────────── próxima rodada ◄────────────────────┘
   │  modelo chama emitir_parecer   (ou o runtime força, ao atingir um limite)
   ▼
validar parecer ──► inválido? problemas registrados e revisão humana obrigatória
   ▼
guardrail de segurança (camada reativa)       [código]
   ▼
agir(): grava na fila de manutenção           [código]
```

## 7.1 Limites de execução

In [ ]:
MAX_RODADAS = 6                  # chamadas ao modelo por triagem, incluindo a que emite o parecer
MAX_CHAMADAS_FERRAMENTA = 12      # chamadas de ferramenta por triagem
ARQUIVO_FILA = Path("fila_manutencao.csv")

print(f"Limites por triagem: {MAX_RODADAS} rodadas · {MAX_CHAMADAS_FERRAMENTA} chamadas de ferramenta")

## 7.2 Percepção

A função `perceber()` prepara as informações iniciais da ordem de serviço que serão utilizadas pelo agente. Tudo aqui é determinístico:

- **Validação da OS.** Sem relato, com relato curto demais ou com obra inexistente, a OS é rejeitada **sem chamar o modelo** — não há o que triar, e não se gasta cota.
- **Códigos de alarme** citados no relato são extraídos por expressão regular.
- **Resolução do equipamento.** Se o prefixo informado existe no cadastro, o agente já recebe tipo, modelo, criticidade, horímetro e ficha técnica, sem gastar uma rodada consultando o cadastro. Se não existe, o contexto diz isso explicitamente, e cabe ao agente decidir o que fazer.
- **Gatilhos de segurança.** É a **camada reativa** do agente híbrido (Aula 1): termos e alarmes que, pelo PRO-SEG-001, implicam bloqueio obrigatório. Quando presentes, a recomendação final não pode ser "liberar", independentemente do que o modelo concluir (seção 7.5). Os padrões são específicos de propósito: *"fumaça escura no escapamento"* não é gatilho; *"fumaça na cabine"* é.

In [ ]:
GATILHOS_SEGURANCA: dict[str, str] = {
    "falha ou anormalidade em freio": r"\bfrei(o|os|ar|ando|ada)\b|\bfrenagem\b",
    "falha na direção": r"\bdirecao\b|\bvolante (duro|pesado|com folga|travad)",
    "fumaça na cabine / cheiro de queimado": r"fumaca (na|dentro da) cabine|cheiro de queimad",
    "indício de incêndio": r"\bincendio\b|\bfogo\b|\bchamas?\b",
    "vazamento de combustível": r"vazamento de (combustivel|diesel|oleo diesel)",
    "vazamento hidráulico com jato sob pressão": r"\bjato\b",
    "trinca estrutural": r"\btrinca",
}
# Alarmes de nível 3 do CAT-ALM-001. A camada reativa não depende do RAG: é uma
# trava fixa, como um intertravamento de segurança em uma máquina.
ALARMES_NIVEL_3 = {"TL-120", "TL-331", "TL-350"}
RE_ALARME = re.compile(r"\bTL-\d{3}\b", re.IGNORECASE)


def _sem_acentos(texto: str) -> str:
    decomposto = unicodedata.normalize("NFKD", texto.lower())
    return "".join(c for c in decomposto if not unicodedata.combining(c))


@dataclass
class Percepcao:
    os_id: str
    valida: bool
    motivo_rejeicao: str | None
    contexto: dict[str, Any]
    equipamento: dict[str, Any] | None
    codigos_alarme: list[str]
    gatilhos: list[str]


def perceber(os_linha: pd.Series) -> Percepcao:
    """Valida a OS e monta o contexto inicial do agente."""
    os_id = str(os_linha.get("os_id", "")).strip() or "SEM-ID"
    relato = str(os_linha.get("relato", "")).strip()
    obra_id = str(os_linha.get("obra_id", "")).strip()
    obras = TABELAS["obras"].set_index("obra_id")

    motivo = None
    if not relato:
        motivo = "OS sem relato do problema."
    elif len(relato) < 15:
        motivo = "Relato curto demais para qualquer triagem."
    elif obra_id not in obras.index:
        motivo = f"Obra '{obra_id}' inexistente no cadastro."
    if motivo:
        return Percepcao(os_id, False, motivo, {}, None, [], [])

    codigos = sorted({c.upper() for c in RE_ALARME.findall(relato)})
    texto = _sem_acentos(relato)
    gatilhos = [nome for nome, padrao in GATILHOS_SEGURANCA.items() if re.search(padrao, texto)]
    gatilhos += [f"alarme de nível 3 ({c})" for c in codigos if c in ALARMES_NIVEL_3]

    informado = str(os_linha.get("equipamento_informado", "")).strip()
    linha_eq = _localizar_equipamento(informado) if informado else None
    equipamento = _resumo_equipamento(linha_eq) if linha_eq is not None else None

    contexto = {
        "os_id": os_id,
        "data_abertura": os_linha.get("data_abertura"),
        "obra": {"obra_id": obra_id, "nome": obras.loc[obra_id, "nome"]},
        "aberta_por": os_linha.get("aberta_por"),
        "relato": relato,
        "equipamento_informado": informado or "(não informado)",
        "equipamento_no_cadastro": equipamento or (
            "NÃO ENCONTRADO: o identificador informado não existe no cadastro da frota."
            if informado else "NÃO INFORMADO"),
        "codigos_de_alarme_no_relato": codigos,
        "gatilhos_de_seguranca_detectados": gatilhos,
    }
    if equipamento and equipamento["obra_id"] != obra_id:
        contexto["divergencia"] = (f"O cadastro situa {equipamento['equipamento_id']} na obra "
                                   f"{equipamento['obra_id']}, mas a OS foi aberta na {obra_id}.")
    return Percepcao(os_id, True, None, contexto, equipamento, codigos, gatilhos)


for _, os_linha in TABELAS["ordens_servico"].iterrows():
    p = perceber(os_linha)
    situacao = ("REJEITADA — " + p.motivo_rejeicao) if not p.valida else (
        f"equipamento {'resolvido: ' + p.equipamento['equipamento_id'] if p.equipamento else 'NÃO resolvido'}"
        f" · alarmes {p.codigos_alarme or '—'} · gatilhos {p.gatilhos or '—'}")
    print(f"{p.os_id}: {situacao}")

## 7.3 Instruções do agente

As instruções do agente definem seu papel, as etapas da análise e as regras que devem ser seguidas durante a triagem. O conteúdo técnico fica nos documentos consultados, enquanto o prompt orienta o agente sobre como utilizar essas informações. As regras principais são:

- **Fato só com ferramenta.** Recorrência, situação da preventiva e disponibilidade de peças só podem ser afirmadas se a ferramenta correspondente foi consultada — e a validação da seção 7.5 confere isso.
- **Consultas em paralelo.** Pedir na mesma rodada as consultas independentes entre si reduz o número de chamadas ao modelo, o recurso mais escasso no free tier.
- **Conclusão por ferramenta.** O modelo opera em modo `ANY`: toda resposta precisa ser uma chamada de função. A triagem só termina com `emitir_parecer`, nunca com texto livre.

In [ ]:
INSTRUCOES_SISTEMA = f"""Você é o agente de triagem de ordens de serviço (OS) de manutenção da Construtora Apollo S.A.
Sua tarefa: investigar a OS recebida e emitir um parecer que oriente o planejador de manutenção.
Você RECOMENDA; não executa reparos, não bloqueia nem libera equipamentos. A data de hoje é {DATA_REFERENCIA:%d/%m/%Y}.

FONTES
- Regras e conhecimento técnico (significado de alarmes, diagnóstico, peças associadas, critérios de bloqueio,
  matriz de prioridade, falha recorrente, devolução de OS): buscar_documentacao_tecnica.
- Fatos operacionais: consultar_equipamento, consultar_historico_manutencao, consultar_plano_preventiva,
  consultar_estoque_pecas, consultar_frota_disponivel.

PROCESSO
1. Equipamento: se o contexto não trouxer o equipamento no cadastro, use consultar_equipamento. Sem identificação
   inequívoca, DEVOLVA a OS (POL-MAN-001 § 3). Nunca escolha um equipamento por aproximação.
2. Relato: se ele não permitir direcionar a investigação, consulte a ficha técnica para saber o que perguntar
   e DEVOLVA a OS com perguntas objetivas.
3. Investigação: consulte a documentação sobre o sintoma ou alarme (pergunta específica, informando ficha_tecnica),
   o histórico (o que já foi feito e se há recorrência) e a preventiva quando o sintoma puder ter relação com ela.
   Verifique no estoque as peças APL que a documentação associar ao problema.
4. Prioridade: recupere a matriz de prioridade e, se houver repetição, a regra de falha recorrente da POL-MAN-001.
   Para equipamento de criticidade A ou parada provável, verifique substituto com consultar_frota_disponivel e,
   se houver algum disponível em QUALQUER obra, informe-o em equipamento_substituto (POL-MAN-001 § 9).
5. Segurança: se houver gatilho de segurança no contexto ou na documentação, recupere o PRO-SEG-001 e recomende
   bloquear_ate_inspecao.

REGRAS
- Peça na MESMA rodada todas as consultas independentes entre si.
- Na documentação, UMA pergunta por assunto (diagnóstico do alarme; regra de recorrência; matriz de prioridade;
  critérios de bloqueio). Perguntas que misturam assuntos trazem trechos genéricos.
- Cada trecho recuperado traz a seção inteira: os códigos APL estão na linha "Peças associadas". Não busque de
  novo o que já foi recuperado.
- Nunca afirme fato operacional sem a ferramenta correspondente: sem histórico, falha_recorrente = nao_verificado;
  sem preventiva, situacao_preventiva = nao_verificada; sem estoque, disponibilidade = nao_verificada.
- fontes_tecnicas: apenas referências efetivamente recuperadas, no formato exato devolvido (ex.: "FTM-CAT320 § 5.2").
- Não liste em pecas_provaveis componente já trocado em corretiva recente do mesmo problema sem diagnóstico que
  justifique nova troca (POL-MAN-001 § 6).
- Registre em pendencias tudo o que não foi possível verificar, inclusive erros de ferramenta.
- Termine SEMPRE chamando emitir_parecer exatamente uma vez. Nunca responda em texto livre.
- Limites desta triagem: {MAX_RODADAS} rodadas de decisão e {MAX_CHAMADAS_FERRAMENTA} chamadas de ferramenta."""

print(f"Instruções de sistema: {len(INSTRUCOES_SISTEMA)} caracteres.")

## 7.4 O parecer: ferramenta terminal e schema

O parecer é o argumento de `emitir_parecer`. Valores ausentes são representados por enumerações explícitas (`nao_verificado`, `nao_verificada`, `NAO_IDENTIFICADO`) em vez de nulos: o modelo precisa *escolher* declarar a ausência, e a validação consegue conferir essa escolha.

In [ ]:
PARECER_SCHEMA: dict[str, Any] = {
    "type": "object",
    "properties": {
        "status_triagem": {"type": "string", "enum": ["concluida", "devolvida"],
                           "description": "devolvida quando o equipamento não está identificado ou o relato é insuficiente."},
        "equipamento_id": {"type": "string",
                           "description": "Prefixo do equipamento no cadastro, ou NAO_IDENTIFICADO."},
        "sistema_afetado": {"type": "string", "enum": SISTEMAS + ["Indeterminado"]},
        "diagnostico_provavel": {"type": "string", "description": "Hipótese principal, em até três frases."},
        "confianca": {"type": "string", "enum": ["alta", "media", "baixa"]},
        "procedimento_recomendado": {"type": "array", "maxItems": 8, "items": {"type": "string"},
                                     "description": "Passos iniciais para o mecânico, segundo a documentação."},
        "pecas_provaveis": {
            "type": "array", "maxItems": 8,
            "items": {
                "type": "object",
                "properties": {
                    "codigo_peca": {"type": "string", "pattern": "^APL-[0-9]{5}$"},
                    "descricao": {"type": "string"},
                    "disponibilidade": {"type": "string", "enum": [
                        "na_obra", "em_outro_almoxarifado", "sem_estoque", "nao_verificada"]},
                },
                "required": ["codigo_peca", "descricao", "disponibilidade"],
                "additionalProperties": False,
            },
        },
        "falha_recorrente": {"type": "string", "enum": ["sim", "nao", "nao_verificado"]},
        "situacao_preventiva": {"type": "string", "enum": ["em_dia", "proxima", "vencida", "nao_verificada"]},
        "prioridade": {"type": "string", "enum": ["P1", "P2", "P3", "P4"]},
        "recomendacao_operacional": {"type": "string", "enum": [
            "liberar", "liberar_com_restricao", "bloquear_ate_inspecao"]},
        "equipamento_substituto": {"type": "string",
                                   "description": "Prefixo do substituto disponível, se verificado; senão vazio."},
        "justificativa": {"type": "string", "description": "Por que esta prioridade e esta recomendação."},
        "fontes_tecnicas": {"type": "array", "items": {"type": "string"}},
        "pendencias": {"type": "array", "items": {"type": "string"}},
        "perguntas_ao_solicitante": {"type": "array", "items": {"type": "string"}},
    },
    "required": ["status_triagem", "equipamento_id", "sistema_afetado", "diagnostico_provavel", "confianca",
                 "procedimento_recomendado", "pecas_provaveis", "falha_recorrente", "situacao_preventiva",
                 "prioridade", "recomendacao_operacional", "justificativa", "fontes_tecnicas",
                 "pendencias", "perguntas_ao_solicitante"],
    "additionalProperties": False,
}
Draft202012Validator.check_schema(PARECER_SCHEMA)
VALIDADOR_PARECER = Draft202012Validator(PARECER_SCHEMA)

DECLARACAO_PARECER = types.FunctionDeclaration(
    name="emitir_parecer",
    description=("Emite o parecer final da triagem e ENCERRA a investigação. Chame exatamente uma vez, quando "
                 "tiver evidência suficiente para concluir ou para devolver a OS ao solicitante."),
    parameters_json_schema=_schema_para_gemini(PARECER_SCHEMA),
)
DECLARACOES_AGENTE = DECLARACOES_CONSULTA + [DECLARACAO_PARECER]
print(f"{len(DECLARACOES_AGENTE)} funções disponíveis ao agente: {[d.name for d in DECLARACOES_AGENTE]}")

## 7.5 Estado, validação do parecer e guardrail de segurança

O `EstadoTriagem` mantém as informações necessárias durante a execução da ordem de serviço, como o histórico da conversa, o número de rodadas e de chamadas de ferramenta e a trajetória das ferramentas utilizadas. Ele também registra quais ferramentas responderam com sucesso, quais referências da documentação foram recuperadas e quais peças tiveram o estoque consultado — informações usadas na validação do parecer.

A validação acontece em duas camadas:

| Camada | O que confere |
|---|---|
| Schema | Campos obrigatórios, tipos e enumerações |
| Domínio | Fontes citadas foram recuperadas · recorrência, preventiva e estoque só afirmados com a ferramenta correspondente · equipamento coerente com o cadastro · devolução acompanhada de perguntas · conclusão sustentada por ao menos uma fonte técnica |

Depois que o parecer é gerado, o código verifica se alguma regra de segurança precisa ser aplicada. Quando uma condição de bloqueio é identificada, o parecer é ajustado conforme a regra implementada, que é o **guardrail de segurança**: com gatilho de segurança na percepção, a recomendação vira `bloquear_ate_inspecao` e a prioridade vira P1 (POL-MAN-001 § 4), mesmo que o modelo tenha concluído outra coisa. A sobrescrita fica registrada no parecer, para auditoria.

In [ ]:
@dataclass
class EstadoTriagem:
    percepcao: Percepcao
    contents: list[types.Content] = field(default_factory=list)
    rodadas: int = 0
    chamadas_ferramenta: int = 0
    ferramentas_com_sucesso: set[str] = field(default_factory=set)
    referencias_recuperadas: set[str] = field(default_factory=set)
    pecas_consultadas: set[str] = field(default_factory=set)
    trajetoria: list[dict[str, Any]] = field(default_factory=list)
    finalizacao: str = ""

    def observar(self, nome: str, resultado: dict[str, Any]) -> None:
        """Atualiza o que foi efetivamente observado a partir de um retorno de ferramenta."""
        if resultado.get("status") != "success":
            return
        self.ferramentas_com_sucesso.add(nome)
        if nome == "buscar_documentacao_tecnica":
            self.referencias_recuperadas.update(e["referencia"] for e in resultado["evidencias"])
        if nome == "consultar_estoque_pecas":
            self.pecas_consultadas.update(p["codigo_peca"] for p in resultado["pecas"])


def validar_parecer(parecer: dict[str, Any], estado: EstadoTriagem) -> list[str]:
    """Devolve a lista de problemas do parecer (vazia = válido)."""
    problemas = _formatar_erros_schema(VALIDADOR_PARECER, parecer)
    if problemas:
        return problemas

    ids_cadastro = set(TABELAS["equipamentos"]["equipamento_id"])
    eq_percebido = estado.percepcao.equipamento
    eq_parecer = parecer["equipamento_id"]

    fontes_invalidas = [f for f in parecer["fontes_tecnicas"] if f not in estado.referencias_recuperadas]
    if fontes_invalidas:
        problemas.append(f"fontes_tecnicas citadas sem terem sido recuperadas: {fontes_invalidas}")
    if parecer["falha_recorrente"] != "nao_verificado" and \
            "consultar_historico_manutencao" not in estado.ferramentas_com_sucesso:
        problemas.append("falha_recorrente afirmada sem consulta bem-sucedida ao histórico")
    if parecer["situacao_preventiva"] != "nao_verificada" and \
            "consultar_plano_preventiva" not in estado.ferramentas_com_sucesso:
        problemas.append("situacao_preventiva afirmada sem consulta bem-sucedida ao plano de preventiva")
    for peca in parecer["pecas_provaveis"]:
        if peca["disponibilidade"] != "nao_verificada" and peca["codigo_peca"] not in estado.pecas_consultadas:
            problemas.append(f"disponibilidade de {peca['codigo_peca']} afirmada sem consulta ao estoque")
    if eq_parecer != "NAO_IDENTIFICADO" and eq_parecer not in ids_cadastro:
        problemas.append(f"equipamento_id {eq_parecer} não existe no cadastro")
    if eq_percebido and eq_parecer != eq_percebido["equipamento_id"]:
        problemas.append(f"equipamento_id deveria ser {eq_percebido['equipamento_id']} (identificado na OS)")
    if parecer["status_triagem"] == "devolvida" and not parecer["perguntas_ao_solicitante"]:
        problemas.append("OS devolvida sem perguntas_ao_solicitante")
    if parecer["status_triagem"] == "concluida":
        if eq_parecer == "NAO_IDENTIFICADO":
            problemas.append("triagem concluída sem equipamento identificado: deveria ser devolvida")
        if not parecer["fontes_tecnicas"]:
            problemas.append("triagem concluída sem nenhuma fonte técnica que sustente o diagnóstico")
    return problemas


def aplicar_guardrail(parecer: dict[str, Any], percepcao: Percepcao) -> list[str]:
    """Camada reativa: gatilho de segurança impõe bloqueio e P1. Devolve as sobrescritas feitas."""
    sobrescritas = []
    if percepcao.gatilhos:
        if parecer.get("recomendacao_operacional") != "bloquear_ate_inspecao":
            sobrescritas.append(f"recomendacao_operacional: {parecer.get('recomendacao_operacional')} "
                                "→ bloquear_ate_inspecao (gatilho de segurança, PRO-SEG-001 § 3)")
            parecer["recomendacao_operacional"] = "bloquear_ate_inspecao"
        if parecer.get("prioridade") != "P1":
            sobrescritas.append(f"prioridade: {parecer.get('prioridade')} → P1 (POL-MAN-001 § 4)")
            parecer["prioridade"] = "P1"
    return sobrescritas


def parecer_incompleto(estado: EstadoTriagem, motivo: str) -> dict[str, Any]:
    """Parecer determinístico quando a triagem não pôde ser concluída pelo modelo.
    Preserva o que foi observado até a falha."""
    p = estado.percepcao
    return {
        "status_triagem": "incompleta",
        "equipamento_id": p.equipamento["equipamento_id"] if p.equipamento else "NAO_IDENTIFICADO",
        "sistema_afetado": "Indeterminado",
        "diagnostico_provavel": "Não determinado: a triagem automática foi interrompida.",
        "confianca": "baixa",
        "procedimento_recomendado": [],
        "pecas_provaveis": [],
        "falha_recorrente": "nao_verificado",
        "situacao_preventiva": "nao_verificada",
        "prioridade": "P1" if p.gatilhos else "P2",
        "recomendacao_operacional": "bloquear_ate_inspecao" if p.gatilhos else "liberar_com_restricao",
        "equipamento_substituto": "",
        "justificativa": f"Triagem interrompida: {motivo.rstrip('.')}. Realizar a triagem manualmente.",
        "fontes_tecnicas": sorted(estado.referencias_recuperadas),
        "pendencias": [motivo, "Triagem manual obrigatória."],
        "perguntas_ao_solicitante": [],
    }


print("Estado, validação, guardrail e parecer de contingência definidos.")

## 7.6 O laço de decisão

O laço implementa o ReAct com chamadas estruturadas: a cada rodada o modelo pede uma ou mais ferramentas, o runtime executa pelo portão da seção 6.4 e devolve as observações. Antes de executar, o runtime aplica as regras de controle:

| Situação | Resposta do runtime |
|---|---|
| Limite de chamadas de ferramenta atingido | Não executa; orienta a concluir |
| Última rodada disponível ou limite de ferramentas atingido | **Finalização forçada**: só `emitir_parecer` fica disponível (`allowed_function_names`) |
| Resposta sem chamada de função ou vazia | Pede ao modelo que responda chamando uma função |
| Parecer inválido | Os problemas ficam registrados e o parecer segue marcado para revisão humana |
| Cota esgotada, erro da API ou falha de rede persistente | Parecer de contingência `incompleta`, com as evidências preservadas |

Durante os testes com o Gemini 3, foi necessário adaptar o tratamento das respostas das ferramentas. Os resultados são enviados novamente no histórico como `user`, e a resposta recebida do modelo é mantida no histórico sem ser reconstruída, o que preserva os metadados de raciocínio que acompanham as chamadas de função.

In [ ]:
def _resumir_resultado(nome: str, r: dict[str, Any]) -> str:
    """Uma linha legível sobre o resultado de uma ferramenta, para o rastro da execução."""
    if r["status"] != "success":
        return f"{r['status']}: {r.get('mensagem', '')[:110]}"
    if nome == "buscar_documentacao_tecnica":
        return "success: " + " | ".join(e["referencia"] for e in r["evidencias"])
    if nome == "consultar_equipamento":
        return "success: " + (r["equipamento"]["equipamento_id"] if "equipamento" in r
                              else f"{r['quantidade']} candidato(s)")
    if nome == "consultar_historico_manutencao":
        return (f"success: {r['total_ocorrencias']} ocorrência(s) · por sistema em 90 dias: "
                f"{r['corretivas_por_sistema_ultimos_90_dias']}")
    if nome == "consultar_plano_preventiva":
        return f"success: vencidas {r['revisoes_vencidas']} · próximas {r['revisoes_proximas']}"
    if nome == "consultar_estoque_pecas":
        return "success: " + " | ".join(f"{p['codigo_peca']} {p['situacao']}" for p in r["pecas"])
    if nome == "consultar_frota_disponivel":
        return f"success: {[e['equipamento_id'] for e in r['equipamentos']] or 'nenhum disponível'}"
    return r["status"]


class AgenteTriagem:
    """Agente único de triagem de OS: percepção, laço ReAct com Tool Calling, validação e ação."""

    def __init__(self, llm, declaracoes: list[types.FunctionDeclaration] = DECLARACOES_AGENTE,
                 ferramentas: dict[str, Ferramenta] = FERRAMENTAS, verbose: bool = True,
                 max_rodadas: int = MAX_RODADAS, max_chamadas: int = MAX_CHAMADAS_FERRAMENTA,
                 atuador: Callable[[dict], None] | None = None):
        self.llm = llm
        self.atuador = atuador or agir      # injetável: os testes usam um atuador que não grava na fila
        self.declaracoes = declaracoes
        self.ferramentas = ferramentas
        self.verbose = verbose
        self.max_rodadas, self.max_chamadas = max_rodadas, max_chamadas

    # ------------------------------------------------------------------ util
    def _log(self, texto: str = "") -> None:
        if self.verbose:
            print(texto)

    def _config(self, apenas_parecer: bool = False) -> types.GenerateContentConfig:
        nomes = ["emitir_parecer"] if apenas_parecer else [d.name for d in self.declaracoes]
        return types.GenerateContentConfig(
            system_instruction=INSTRUCOES_SISTEMA,
            tools=[types.Tool(function_declarations=self.declaracoes)],
            tool_config=types.ToolConfig(function_calling_config=types.FunctionCallingConfig(
                mode=types.FunctionCallingConfigMode.ANY, allowed_function_names=nomes)),
            automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
        )

    def _rodada(self, estado: EstadoTriagem, apenas_parecer: bool) -> list[types.FunctionCall]:
        """Uma chamada ao modelo. Anexa o conteúdo devolvido ao histórico, sem reconstruí-lo."""
        estado.rodadas += 1
        resposta = self.llm.gerar(estado.contents, self._config(apenas_parecer))
        candidato = resposta.candidates[0] if resposta.candidates else None
        if candidato is None or candidato.content is None or not candidato.content.parts:
            return []
        estado.contents.append(candidato.content)
        return list(resposta.function_calls or [])

    def _executar_chamada(self, estado: EstadoTriagem, chamada: types.FunctionCall) -> dict[str, Any]:
        """Aplica o limite de chamadas e executa a ferramenta pelo portão da seção 6.4."""
        nome, argumentos = chamada.name, dict(chamada.args or {})
        if estado.chamadas_ferramenta >= self.max_chamadas:
            resultado = {"status": "error", "mensagem": "Limite de chamadas de ferramenta atingido. "
                                                         "Conclua com emitir_parecer."}
        else:
            estado.chamadas_ferramenta += 1
            resultado = executar_ferramenta(nome, argumentos, registro=self.ferramentas)
            estado.observar(nome, resultado)
        estado.trajetoria.append({"rodada": estado.rodadas, "ferramenta": nome, "argumentos": argumentos,
                                  "status": resultado["status"]})
        self._log(f"   → {nome}({', '.join(f'{k}={v!r}' for k, v in argumentos.items())})")
        self._log(f"     {_resumir_resultado(nome, resultado)}")
        return resultado

    @staticmethod
    def _texto_ao_modelo(estado: EstadoTriagem, texto: str) -> None:
        """Acrescenta uma instrução do runtime ao turno do usuário, sem criar dois turnos seguidos."""
        if estado.contents and estado.contents[-1].role == "user":
            estado.contents[-1].parts.append(types.Part(text=texto))
        else:
            estado.contents.append(types.Content(role="user", parts=[types.Part(text=texto)]))

    @staticmethod
    def _resposta_funcao(chamada: types.FunctionCall, resultado: dict[str, Any]) -> types.Part:
        return types.Part(function_response=types.FunctionResponse(
            id=chamada.id, name=chamada.name, response=resultado))

    # ------------------------------------------------------------------ laço
    def _decidir(self, estado: EstadoTriagem) -> dict[str, Any]:
        """Laço ReAct até o modelo chamar emitir_parecer ou o runtime forçar a finalização."""
        while True:
            forcar = (estado.rodadas >= self.max_rodadas - 1
                      or estado.chamadas_ferramenta >= self.max_chamadas)
            if forcar:
                self._texto_ao_modelo(estado, "Limite de execução atingido. Emita AGORA o parecer com as "
                                      "evidências já coletadas, registrando em pendencias o que não foi verificado.")
                estado.finalizacao = "forcada_por_limite"
                self._log(f"\n[LIMITE] Rodada {estado.rodadas + 1}: limite atingido — finalização forçada.")
            else:
                self._log(f"\nRodada {estado.rodadas + 1}")

            chamadas = self._rodada(estado, apenas_parecer=forcar)
            if not chamadas:
                if forcar:
                    raise RuntimeError("o modelo não emitiu o parecer nem na finalização forçada")
                self._texto_ao_modelo(estado, "Responda chamando uma das funções disponíveis; para "
                                      "concluir, chame emitir_parecer.")
                self._log("   (resposta sem chamada de função — o modelo foi orientado a usar as funções)")
                continue

            parecer_chamada = next((c for c in chamadas if c.name == "emitir_parecer"), None)
            if parecer_chamada is None and forcar:
                raise RuntimeError("o modelo não emitiu o parecer na finalização forçada")
            if parecer_chamada is not None:
                outras = [c.name for c in chamadas if c.name != "emitir_parecer"]
                if outras:
                    self._log(f"   (chamadas descartadas por virem junto do parecer: {outras})")
                estado.finalizacao = estado.finalizacao or "natural"
                self._log("   → emitir_parecer(...)")
                estado.trajetoria.append({"rodada": estado.rodadas, "ferramenta": "emitir_parecer",
                                          "argumentos": {}, "status": "emitido"})
                return dict(parecer_chamada.args or {})

            self._log(f"   O modelo pediu {len(chamadas)} ferramenta(s):")
            respostas = [self._resposta_funcao(c, self._executar_chamada(estado, c)) for c in chamadas]
            estado.contents.append(types.Content(role="user", parts=respostas))

    # ------------------------------------------------------------ ciclo completo
    def executar(self, os_linha: pd.Series) -> dict[str, Any]:
        """Ciclo perceber → decidir → validar → guardrail → agir para uma OS."""
        percepcao = perceber(os_linha)
        self._log("=" * 78)
        self._log(f"TRIAGEM {percepcao.os_id}")
        self._log("=" * 78)

        if not percepcao.valida:
            self._log(f"OS rejeitada na percepção, sem chamar o modelo: {percepcao.motivo_rejeicao}")
            registro = self._registro(percepcao, None, None, [], [], rejeitada=True)
            registro["modelo"] = "—"
            self.atuador(registro)
            return registro

        self._log(f"Relato: {percepcao.contexto['relato']}")
        self._log(f"Equipamento: {percepcao.equipamento['equipamento_id'] if percepcao.equipamento else 'NÃO resolvido'}"
                  f" · alarmes: {percepcao.codigos_alarme or '—'} · gatilhos: {percepcao.gatilhos or '—'}")

        estado = EstadoTriagem(percepcao=percepcao)
        estado.contents.append(types.Content(role="user", parts=[types.Part(text=(
            "Ordem de serviço para triagem:\n"
            + json.dumps(percepcao.contexto, ensure_ascii=False, indent=2)))]))

        problemas: list[str] = []
        try:
            parecer = self._decidir(estado)
            problemas = validar_parecer(parecer, estado)
            if problemas:
                self._log(f"\n[VALIDAÇÃO] Parecer com {len(problemas)} problema(s); vai para revisão humana:")
                for problema in problemas:
                    self._log(f"   - {problema}")
        except (CotaEsgotadaError, errors.APIError, httpx.HTTPError, RuntimeError) as erro:
            motivo = f"{type(erro).__name__}: {erro}"
            self._log(f"\n[CONTINGÊNCIA] Triagem interrompida — {motivo}")
            parecer, estado.finalizacao, problemas = parecer_incompleto(estado, motivo), "contingencia", []

        sobrescritas = aplicar_guardrail(parecer, percepcao)
        for s in sobrescritas:
            self._log(f"\n[GUARDRAIL] {s}")
        registro = self._registro(percepcao, estado, parecer, problemas, sobrescritas)
        registro["modelo"] = getattr(self.llm, "modelo", "simulado")
        self.atuador(registro)
        self._log(f"\nResultado: {registro['status_triagem']} · prioridade {registro['prioridade']} · "
                  f"{registro['recomendacao_operacional']} · {estado.rodadas} rodada(s) · "
                  f"{estado.chamadas_ferramenta} ferramenta(s) · finalização: {estado.finalizacao}"
                  + (" · REVISÃO HUMANA OBRIGATÓRIA" if registro["revisao_humana_obrigatoria"] else ""))
        if hasattr(self.llm, "resumo"):
            self._log(self.llm.resumo())
        return registro

    @staticmethod
    def _registro(percepcao: Percepcao, estado: EstadoTriagem | None, parecer: dict[str, Any] | None,
                  problemas: list[str], sobrescritas: list[str], rejeitada: bool = False) -> dict[str, Any]:
        base = {"os_id": percepcao.os_id, "data_triagem": DATA_REFERENCIA.isoformat()}
        if rejeitada:
            return base | {"status_triagem": "rejeitada", "equipamento_id": "", "prioridade": "",
                           "recomendacao_operacional": "", "justificativa": percepcao.motivo_rejeicao,
                           "rodadas": 0, "chamadas_ferramenta": 0, "ferramentas_usadas": [],
                           "finalizacao": "rejeitada_na_percepcao", "revisao_humana_obrigatoria": True,
                           "problemas_validacao": [], "sobrescritas": [], "gatilhos": [], "trajetoria": [],
                           "referencias_recuperadas": [], "parecer": None}
        return base | {
            "status_triagem": parecer["status_triagem"],
            "equipamento_id": parecer["equipamento_id"],
            "prioridade": parecer["prioridade"],
            "recomendacao_operacional": parecer["recomendacao_operacional"],
            "justificativa": parecer["justificativa"],
            "rodadas": estado.rodadas,
            "chamadas_ferramenta": estado.chamadas_ferramenta,
            "ferramentas_usadas": sorted({t["ferramenta"] for t in estado.trajetoria}),
            "finalizacao": estado.finalizacao,
            "revisao_humana_obrigatoria": bool(problemas) or parecer["status_triagem"] == "incompleta",
            "problemas_validacao": problemas,
            "sobrescritas": sobrescritas,
            "gatilhos": percepcao.gatilhos,
            "trajetoria": estado.trajetoria,
            "referencias_recuperadas": sorted(estado.referencias_recuperadas),
            "parecer": parecer,
        }


print("AgenteTriagem definido.")

## 7.7 Ação: a fila de manutenção

`agir()` é o atuador do agente (PEAS): grava o parecer em `fila_manutencao.csv`, que o planejador consome. A gravação é idempotente por `os_id` — retriar uma OS substitui a linha anterior em vez de duplicá-la.

In [ ]:
FILA: dict[str, dict[str, Any]] = {}


def agir(registro: dict[str, Any]) -> None:
    """Registra o parecer na fila de manutenção (atuador do agente)."""
    p = registro.get("parecer") or {}
    FILA[registro["os_id"]] = {
        "os_id": registro["os_id"],
        "data_triagem": registro["data_triagem"],
        "status_triagem": registro["status_triagem"],
        "equipamento_id": registro["equipamento_id"],
        "prioridade": registro["prioridade"],
        "recomendacao_operacional": registro["recomendacao_operacional"],
        "sistema_afetado": p.get("sistema_afetado", ""),
        "diagnostico_provavel": p.get("diagnostico_provavel", ""),
        "confianca": p.get("confianca", ""),
        "falha_recorrente": p.get("falha_recorrente", ""),
        "situacao_preventiva": p.get("situacao_preventiva", ""),
        "procedimento_recomendado": " | ".join(p.get("procedimento_recomendado", [])),
        "pecas_provaveis": json.dumps(p.get("pecas_provaveis", []), ensure_ascii=False),
        "equipamento_substituto": p.get("equipamento_substituto", ""),
        "fontes_tecnicas": "; ".join(p.get("fontes_tecnicas", [])),
        "pendencias": " | ".join(p.get("pendencias", [])),
        "perguntas_ao_solicitante": " | ".join(p.get("perguntas_ao_solicitante", [])),
        "justificativa": registro["justificativa"],
        "revisao_humana_obrigatoria": registro["revisao_humana_obrigatoria"],
        "sobrescritas_guardrail": " | ".join(registro["sobrescritas"]),
        "modelo": registro.get("modelo", ""),
        "rodadas": registro["rodadas"],
        "chamadas_ferramenta": registro["chamadas_ferramenta"],
    }
    pd.DataFrame(FILA.values()).to_csv(ARQUIVO_FILA, index=False, encoding="utf-8")


def mostrar_parecer(registro: dict[str, Any]) -> None:
    """Exibe o parecer de forma legível."""
    p = registro["parecer"]
    if p is None:
        print(f"{registro['os_id']}: rejeitada — {registro['justificativa']}")
        return
    linhas = [
        ("Status", p["status_triagem"]), ("Equipamento", p["equipamento_id"]),
        ("Sistema", p["sistema_afetado"]), ("Diagnóstico", f"{p['diagnostico_provavel']} (confiança {p['confianca']})"),
        ("Falha recorrente", p["falha_recorrente"]), ("Preventiva", p["situacao_preventiva"]),
        ("Prioridade", p["prioridade"]), ("Recomendação", p["recomendacao_operacional"]),
        ("Substituto", p.get("equipamento_substituto") or "—"),
        ("Justificativa", p["justificativa"]),
        ("Fontes", "; ".join(p["fontes_tecnicas"]) or "—"),
    ]
    for rotulo, valor in linhas:
        print(f"{rotulo:<17} {valor}")
    for titulo, itens in [("Procedimento", p["procedimento_recomendado"]),
                          ("Peças", [f"{x['codigo_peca']} {x['descricao']} — {x['disponibilidade']}"
                                     for x in p["pecas_provaveis"]]),
                          ("Pendências", p["pendencias"]), ("Perguntas", p["perguntas_ao_solicitante"])]:
        if itens:
            print(f"{titulo}:")
            for item in itens:
                print(f"  - {item}")
    if registro["sobrescritas"]:
        print("Guardrail:", " | ".join(registro["sobrescritas"]))
    if registro["problemas_validacao"]:
        print("Problemas de validação:", " | ".join(registro["problemas_validacao"]))


print(f"Atuador pronto: {ARQUIVO_FILA}")

## 7.8 Demonstração: OS-2026-0101

Primeira triagem com o modelo real. A OS-2026-0101 é o cenário mais rico: alarme TL-214 na escavadeira EH-07, que já teve duas corretivas hidráulicas nos últimos 45 dias. Espera-se que o agente consulte a ficha técnica, perceba a recorrência pelo histórico, recupere a regra de falha recorrente e verifique as peças. O rastro abaixo mostra cada decisão do modelo e cada observação devolvida pelo runtime.

In [ ]:
agente = AgenteTriagem(llm)
ordens = TABELAS["ordens_servico"].set_index("os_id", drop=False)

registros: dict[str, dict[str, Any]] = {}
registros["OS-2026-0101"] = agente.executar(ordens.loc["OS-2026-0101"])

In [ ]:
mostrar_parecer(registros["OS-2026-0101"])

**Comentando o resultado.** Na execução de referência, o agente concluiu a OS-2026-0101 em **3 rodadas e 8 chamadas de ferramenta**. Na primeira rodada pediu seis consultas em paralelo — três buscas na documentação, cada uma sobre um único assunto, mais histórico, preventiva e frota. Na segunda, consultou o estoque dos códigos APL lidos na ficha técnica e o procedimento de segurança. Na terceira, emitiu o parecer, aprovado pela validação de primeira.

Nesse caso, o agente relacionou as informações do histórico com o procedimento técnico. Como as intervenções anteriores (limpeza do arrefecedor e troca do filtro de retorno) não resolveram o problema, a recomendação foi seguir para o item e) da FTM-CAT320 § 5.2: medição das pressões e investigação da possibilidade de vazamento interno. A ocorrência foi caracterizada como falha recorrente (2 anteriores + a atual, POL-MAN-001 § 6).

A revisão crítica dessa execução apontou dois deslizes, que motivaram ajustes:

| Deslize | Ajuste |
|---|---|
| A frota indicou a EH-13 disponível em outra obra, mas o parecer registrou "sem substituto" | As instruções passaram a exigir a indicação de substituto disponível em qualquer obra (POL-MAN-001 § 9) |
| O filtro de retorno, trocado em 09/09, apareceu de novo entre as peças prováveis | Instrução específica: não listar componente já trocado em corretiva recente sem diagnóstico que justifique nova troca (POL-MAN-001 § 6) |


---
# 8. Testes de comportamento

Os testes foram divididos em duas partes. Primeiro, o modelo simulado é utilizado para verificar situações controladas, como erros, limites de chamadas e pareceres inválidos. Depois, os testes com o modelo real verificam o comportamento do agente nos cenários definidos para o projeto. Juntas, as duas partes cobrem as quatro dimensões pedidas no enunciado — **seleção de ferramentas, respeito aos limites, tratamento de ausência de informação e conclusão apropriada** —, além da recuperação diante de falhas e do guardrail de segurança.

| Camada | Modelo | Por quê |
|---|---|---|
| **8.1–8.2 Testes com modelo simulado** | Respostas programadas | Falhas de API, limites estourados e pareceres inválidos não acontecem sob demanda com o modelo real. O simulador força cada situação, de forma **determinística e sem gastar cota** |
| **8.3–8.4 Testes com o modelo real** | `gemini-3.6-flash` ou um modelo de reserva | Só o modelo real mostra se o agente **escolhe** as ferramentas certas e **conclui** corretamente em cada cenário plantado nos dados |

Nos testes com o modelo real, a validação considera características objetivas do resultado, como as ferramentas utilizadas, o status da triagem e os valores das enumerações do parecer. Assim, não é necessário comparar o texto completo do parecer, que pode variar entre execuções.

## 8.1 O modelo simulado

`ModeloSimulado` implementa a mesma interface de `ClienteLLM` (`gerar(contents, config)`), devolvendo respostas programadas no formato real do SDK. O agente não distingue o Gemini do simulador, e por isso os testes exercitam o mesmo código usado com o modelo real. O simulador também registra o que recebeu em cada chamada, o que permite verificar, por exemplo, quais funções o runtime liberou numa finalização forçada.

Para não depender de embeddings, os testes simulados usam uma versão da ferramenta de documentação que devolve trechos fixos da base, e um atuador que não grava na fila de manutenção.

In [ ]:
from dataclasses import replace
from types import SimpleNamespace


class ModeloSimulado:
    """Mesma interface de ClienteLLM.gerar, com respostas programadas."""

    def __init__(self, passos: list[Any]):
        self.passos = list(passos)
        self.chamadas = 0
        self.configs: list[types.GenerateContentConfig] = []
        self.papeis: list[list[str]] = []

    def gerar(self, contents, config):
        self.chamadas += 1
        self.configs.append(config)
        self.papeis.append([c.role for c in contents])
        if not self.passos:
            raise RuntimeError("roteiro do modelo simulado esgotado")
        passo = self.passos.pop(0)
        if isinstance(passo, BaseException):
            raise passo
        return passo


class ClienteGenaiSimulado:
    """Substitui o cliente do SDK dentro do ClienteLLM, para testar retry, cota e modelos de reserva."""

    def __init__(self, passos: list[Any]):
        self.passos = list(passos)
        self.modelos_chamados: list[str] = []
        self.models = SimpleNamespace(generate_content=self._gerar)

    def _gerar(self, model, contents, config):
        self.modelos_chamados.append(model)
        passo = self.passos.pop(0)
        if isinstance(passo, BaseException):
            raise passo
        return passo


def resposta(*chamadas: tuple[str, dict[str, Any]]) -> types.GenerateContentResponse:
    """Resposta do modelo com uma ou mais chamadas de função, no formato do SDK."""
    partes = [types.Part(function_call=types.FunctionCall(id=f"sim-{i}", name=nome, args=args))
              for i, (nome, args) in enumerate(chamadas)]
    return types.GenerateContentResponse(candidates=[types.Candidate(
        content=types.Content(role="model", parts=partes), finish_reason="STOP")])


def parecer_simulado(**alteracoes) -> dict[str, Any]:
    """Parecer válido mínimo (OS devolvida, nada afirmado); cada teste altera o que precisa."""
    base = {
        "status_triagem": "devolvida", "equipamento_id": "EH-07", "sistema_afetado": "Indeterminado",
        "diagnostico_provavel": "Informação insuficiente para diagnóstico.", "confianca": "baixa",
        "procedimento_recomendado": [], "pecas_provaveis": [], "falha_recorrente": "nao_verificado",
        "situacao_preventiva": "nao_verificada", "prioridade": "P3",
        "recomendacao_operacional": "liberar_com_restricao", "equipamento_substituto": "",
        "justificativa": "Parecer de teste.", "fontes_tecnicas": [], "pendencias": ["teste"],
        "perguntas_ao_solicitante": ["Desde quando o sintoma ocorre?"],
    }
    return base | alteracoes


def _documentacao_simulada(pergunta: str, ficha_tecnica: str | None = None) -> dict[str, Any]:
    trechos = [c for c in CHUNKS if c["metadata"]["aplicacao"] in (ficha_tecnica, "geral")][:3]
    return _sucesso(pergunta=pergunta, filtro=ficha_tecnica or "todos os documentos", evidencias=[
        {"referencia": c["metadata"]["referencia"], "documento": c["metadata"]["titulo_documento"],
         "secao": c["metadata"]["titulo"], "trecho": c["texto"]} for c in trechos])


FERRAMENTAS_TESTE = FERRAMENTAS | {
    "buscar_documentacao_tecnica": replace(FERRAMENTAS["buscar_documentacao_tecnica"],
                                           funcao=_documentacao_simulada)}
REF_SIMULADA = _documentacao_simulada("x" * 10, "FTM-CAT320")["evidencias"][0]["referencia"]


def triar_simulado(os_id: str, modelo, ferramentas=FERRAMENTAS_TESTE, **limites):
    """Executa uma triagem com modelo simulado e atuador que não grava na fila."""
    gravados: list[dict] = []
    agente_teste = AgenteTriagem(modelo, ferramentas=ferramentas, verbose=False,
                                 atuador=gravados.append, **limites)
    return agente_teste.executar(ordens.loc[os_id])


RESULTADOS_SIMULADOS: list[dict[str, Any]] = []


def teste_simulado(categoria: str, descricao: str):
    """Decorador que executa o teste na hora e registra OK ou FALHOU."""
    def decorador(funcao):
        try:
            detalhe, ok = funcao() or "", True
        except AssertionError as erro:
            detalhe, ok = f"falhou: {erro}", False
        except Exception as erro:
            detalhe, ok = f"erro inesperado: {type(erro).__name__}: {erro}", False
        RESULTADOS_SIMULADOS.append({"categoria": categoria, "teste": descricao,
                                     "resultado": "OK" if ok else "FALHOU", "detalhe": detalhe})
        return funcao
    return decorador


print("Modelo simulado e utilitários de teste prontos.")

## 8.2 Testes com modelo simulado

Cada teste programa as respostas do modelo, executa o agente real — percepção, laço, portão de ferramentas, validação, guardrail — e confere o comportamento com `assert`. Nenhum deles chama a API.

In [ ]:
e503 = errors.ServerError(503, {"error": {"code": 503, "message": "high demand", "status": "UNAVAILABLE"}})
e429_diario = errors.ClientError(429, {"error": {
    "code": 429, "message": "quota exceeded", "status": "RESOURCE_EXHAUSTED",
    "details": [{"@type": "type.googleapis.com/google.rpc.RetryInfo", "retryDelay": "19144s"}]}})
sem_espera = lambda segundos: None


# ----------------------------------------------------------------- limites
@teste_simulado("Respeito aos limites", "modelo que nunca conclui: finalização forçada na última rodada")
def _():
    passos = [resposta(("consultar_historico_manutencao", {"equipamento_id": "EH-07", "janela_dias": 30 + i}))
              for i in range(MAX_RODADAS - 1)] + [resposta(("emitir_parecer", parecer_simulado()))]
    modelo = ModeloSimulado(passos)
    r = triar_simulado("OS-2026-0101", modelo)
    assert r["rodadas"] == MAX_RODADAS, r["rodadas"]
    assert r["finalizacao"] == "forcada_por_limite", r["finalizacao"]
    liberadas = modelo.configs[-1].tool_config.function_calling_config.allowed_function_names
    assert liberadas == ["emitir_parecer"], liberadas
    return f"{r['rodadas']} rodadas; na última, só {liberadas} foi liberada"


@teste_simulado("Respeito aos limites", f"no máximo {MAX_CHAMADAS_FERRAMENTA} chamadas de ferramenta por triagem")
def _():
    excesso = [("consultar_historico_manutencao", {"equipamento_id": "EH-07", "janela_dias": d})
               for d in range(1, MAX_CHAMADAS_FERRAMENTA + 2)]
    r = triar_simulado("OS-2026-0101", ModeloSimulado(
        [resposta(*excesso), resposta(("emitir_parecer", parecer_simulado()))]))
    assert r["chamadas_ferramenta"] == MAX_CHAMADAS_FERRAMENTA, r["chamadas_ferramenta"]
    assert r["finalizacao"] == "forcada_por_limite", r["finalizacao"]
    recusadas = sum(t["status"] == "error" for t in r["trajetoria"])
    return f"{len(excesso)} pedidas, {r['chamadas_ferramenta']} executadas, {recusadas} recusada(s)"


# ----------------------------------------------------------------- robustez
@teste_simulado("Recuperação de falhas", "argumento inválido, seguido de correção pelo modelo")
def _():
    r = triar_simulado("OS-2026-0101", ModeloSimulado([
        resposta(("consultar_estoque_pecas", {"codigos_peca": ["30021"]})),
        resposta(("consultar_estoque_pecas", {"codigos_peca": ["APL-10021"], "obra_id": "OB-01"})),
        resposta(("emitir_parecer", parecer_simulado()))]))
    status = [t["status"] for t in r["trajetoria"]]
    assert status == ["invalid_arguments", "success", "emitido"], status
    return " → ".join(status)


@teste_simulado("Recuperação de falhas", "exceção dentro de uma ferramenta")
def _():
    def preventiva_quebrada(equipamento_id):
        raise KeyError("coluna_inexistente")
    ferramentas = FERRAMENTAS_TESTE | {"consultar_plano_preventiva": replace(
        FERRAMENTAS["consultar_plano_preventiva"], funcao=preventiva_quebrada)}
    log_antes = len(LOG_RUNTIME)
    r = triar_simulado("OS-2026-0101", ModeloSimulado([
        resposta(("consultar_plano_preventiva", {"equipamento_id": "EH-07"})),
        resposta(("emitir_parecer", parecer_simulado()))]), ferramentas=ferramentas)
    assert r["trajetoria"][0]["status"] == "error"
    assert len(LOG_RUNTIME) == log_antes + 1 and "KeyError" in LOG_RUNTIME[-1]["erro"]
    del LOG_RUNTIME[log_antes:]          # a falha foi provocada pelo teste: não polui o log da execução
    assert r["status_triagem"] == "devolvida"
    return "erro normalizado para o modelo; detalhe técnico só no log; triagem concluída"


@teste_simulado("Recuperação de falhas", "ferramenta inexistente solicitada")
def _():
    r = triar_simulado("OS-2026-0101", ModeloSimulado([
        resposta(("apagar_ordem_de_servico", {"os_id": "OS-2026-0101"})),
        resposta(("emitir_parecer", parecer_simulado()))]))
    assert r["trajetoria"][0]["status"] == "error"
    return "recusada sem execução"


@teste_simulado("Recuperação de falhas", "API sobrecarregada (503) duas vezes, depois sucesso")
def _():
    esperas: list[float] = []
    llm_teste = ClienteLLM(ClienteGenaiSimulado([e503, e503, resposta(("emitir_parecer", parecer_simulado()))]),
                           MODELO, MODELO_EMBEDDING, dormir=esperas.append)
    r = triar_simulado("OS-2026-0101", llm_teste)
    assert len(esperas) == 2 and r["status_triagem"] == "devolvida"
    return f"2 esperas com backoff ({esperas[0]:.0f}s, {esperas[1]:.0f}s) e triagem concluída"


@teste_simulado("Recuperação de falhas", "cota diária esgotada no meio da triagem")
def _():
    llm_teste = ClienteLLM(ClienteGenaiSimulado([
        resposta(("buscar_documentacao_tecnica", {"pergunta": "alarme TL-214 óleo hidráulico quente",
                                                  "ficha_tecnica": "FTM-CAT320"})), e429_diario]),
        MODELO, MODELO_EMBEDDING, dormir=sem_espera)
    r = triar_simulado("OS-2026-0101", llm_teste)
    assert r["status_triagem"] == "incompleta" and r["revisao_humana_obrigatoria"]
    assert r["parecer"]["fontes_tecnicas"], "evidências coletadas deveriam ser preservadas"
    return f"incompleta, sem retry inútil; {len(r['parecer']['fontes_tecnicas'])} evidência(s) preservada(s)"


@teste_simulado("Recuperação de falhas", "após cota esgotada, novas chamadas não chegam à API")
def _():
    cliente_sim = ClienteGenaiSimulado([e429_diario])
    llm_teste = ClienteLLM(cliente_sim, MODELO, MODELO_EMBEDDING, dormir=sem_espera)
    triar_simulado("OS-2026-0101", llm_teste)
    r = triar_simulado("OS-2026-0102", llm_teste)
    assert llm_teste.cota_esgotada and llm_teste.chamadas_geracao == 1
    assert r["status_triagem"] == "incompleta"
    return "2ª triagem recusada localmente, sem nova requisição"


@teste_simulado("Recuperação de falhas", "modelo principal sobrecarregado: triagem segue no modelo de reserva")
def _():
    cliente_sim = ClienteGenaiSimulado([e503] * TENTATIVAS_POR_CHAMADA
                                       + [resposta(("emitir_parecer", parecer_simulado()))])
    llm_teste = ClienteLLM(cliente_sim, ["modelo-principal", "modelo-reserva"], MODELO_EMBEDDING,
                           dormir=sem_espera)
    r = triar_simulado("OS-2026-0101", llm_teste)
    assert cliente_sim.modelos_chamados == ["modelo-principal"] * TENTATIVAS_POR_CHAMADA + ["modelo-reserva"]
    assert r["status_triagem"] == "devolvida" and r["modelo"] == "modelo-reserva"
    return f"{TENTATIVAS_POR_CHAMADA} falhas no principal; triagem concluída no modelo de reserva"


@teste_simulado("Recuperação de falhas", "cota do modelo principal esgotada: reserva assume")
def _():
    cliente_sim = ClienteGenaiSimulado([e429_diario, resposta(("emitir_parecer", parecer_simulado()))])
    llm_teste = ClienteLLM(cliente_sim, ["modelo-principal", "modelo-reserva"], MODELO_EMBEDDING,
                           dormir=sem_espera)
    r = triar_simulado("OS-2026-0101", llm_teste)
    assert r["status_triagem"] == "devolvida" and not llm_teste.cota_esgotada
    return "a cota é por modelo: a triagem continuou no modelo de reserva"


@teste_simulado("Recuperação de falhas", "timeout de rede persistente")
def _():
    llm_teste = ClienteLLM(ClienteGenaiSimulado([httpx.ReadTimeout("timed out")] * TENTATIVAS_POR_CHAMADA),
                           MODELO, MODELO_EMBEDDING, dormir=sem_espera)
    r = triar_simulado("OS-2026-0101", llm_teste)
    assert r["status_triagem"] == "incompleta"
    return f"{TENTATIVAS_POR_CHAMADA} tentativas, depois parecer de contingência"


# ----------------------------------------------------------------- ausência de informação
@teste_simulado("Ausência de informação", "OS sem relato não chama o modelo")
def _():
    modelo = ModeloSimulado([])
    r = triar_simulado("OS-2026-0106", modelo)
    assert modelo.chamadas == 0 and r["status_triagem"] == "rejeitada"
    return "rejeitada na percepção, 0 chamadas"


@teste_simulado("Ausência de informação", "fato operacional afirmado sem a ferramenta é rejeitado")
def _():
    estado = EstadoTriagem(percepcao=perceber(ordens.loc["OS-2026-0101"]))
    problemas = validar_parecer(parecer_simulado(falha_recorrente="sim", situacao_preventiva="vencida",
        pecas_provaveis=[{"codigo_peca": "APL-10021", "descricao": "filtro", "disponibilidade": "na_obra"}]), estado)
    assert len(problemas) == 3, problemas
    return "recorrência, preventiva e estoque sem consulta: 3 problemas apontados"


# ----------------------------------------------------------------- conclusão apropriada
@teste_simulado("Conclusão apropriada", "parecer inválido vai para revisão humana")
def _():
    invalido = parecer_simulado(status_triagem="concluida", fontes_tecnicas=[])
    r = triar_simulado("OS-2026-0101", ModeloSimulado([resposta(("emitir_parecer", invalido))]))
    assert r["revisao_humana_obrigatoria"] and r["problemas_validacao"]
    return "marcado para revisão humana: " + r["problemas_validacao"][0]


# ----------------------------------------------------------------- segurança e protocolo
@teste_simulado("Segurança", "guardrail impede 'liberar' em OS com falha de freio")
def _():
    r = triar_simulado("OS-2026-0105", ModeloSimulado([resposta(("emitir_parecer", parecer_simulado(
        equipamento_id="CR-04", recomendacao_operacional="liberar", prioridade="P3")))]))
    assert r["recomendacao_operacional"] == "bloquear_ate_inspecao" and r["prioridade"] == "P1"
    assert len(r["sobrescritas"]) == 2
    return "liberar → bloquear_ate_inspecao; P3 → P1"


@teste_simulado("Protocolo", "histórico enviado ao modelo nunca usa o papel 'tool'")
def _():
    modelo = ModeloSimulado([
        resposta(("consultar_plano_preventiva", {"equipamento_id": "EH-07"}),
                 ("consultar_historico_manutencao", {"equipamento_id": "EH-07"})),
        resposta(("emitir_parecer", parecer_simulado()))])
    triar_simulado("OS-2026-0101", modelo)
    papeis = modelo.papeis[-1]
    assert set(papeis) <= {"user", "model"}, papeis
    assert all(a != b for a, b in zip(papeis, papeis[1:])), papeis
    return " → ".join(papeis)


with pd.option_context("display.max_colwidth", 110):
    display(pd.DataFrame(RESULTADOS_SIMULADOS))
falhas = [t for t in RESULTADOS_SIMULADOS if t["resultado"] == "FALHOU"]
assert not falhas, f"{len(falhas)} teste(s) simulado(s) falharam"
print(f"{len(RESULTADOS_SIMULADOS)} testes simulados aprovados · {llm.resumo()}")

## 8.3 Triagem das demais ordens com o modelo real

As cinco ordens restantes cobrem os cenários plantados nos dados:

| OS | Cenário | Comportamento esperado |
|---|---|---|
| OS-2026-0102 | "Caminhão fazendo barulho estranho na frente" | Devolver, com as perguntas que a ficha técnica indica |
| OS-2026-0103 | Escavadeira EH-15, que não existe; a obra tem duas escavadeiras | Não escolher por aproximação; devolver |
| OS-2026-0104 | Trator perdendo potência, com R500 vencida | Consultar a preventiva e associá-la ao sintoma |
| OS-2026-0105 | Freio da carregadeira com alarme TL-331; válvula sem estoque | Bloquear, P1, peça sem estoque declarada |
| OS-2026-0106 | Sem relato | Rejeitar sem chamar o modelo |

Se a cota diária do modelo se esgotar durante a execução, as OS seguintes não são enviadas à API e os testes correspondentes aparecem como **não executados**, em vez de falhar.

In [ ]:
for os_id in ["OS-2026-0102", "OS-2026-0103", "OS-2026-0104", "OS-2026-0105", "OS-2026-0106"]:
    if os_id in registros:
        continue
    if perceber(ordens.loc[os_id]).valida and llm.cota_esgotada:
        print(f"\n{os_id}: não triada — cota diária do modelo esgotada.")
        continue
    registros[os_id] = agente.executar(ordens.loc[os_id])
    print()

In [ ]:
# Resumo de execução: como cada triagem terminou
resumo_execucao = pd.DataFrame([{
    "OS": os_id,
    "status": r["status_triagem"],
    "finalização": r["finalizacao"],
    "modelo": r.get("modelo", "—"),
    "rodadas": r["rodadas"],
    "ferramentas": r["chamadas_ferramenta"],
    "revisão humana": r["revisao_humana_obrigatoria"],
    "observação": (r["parecer"]["pendencias"][0] if r["finalizacao"] == "contingencia"
                   else "; ".join(r["problemas_validacao"])[:120]),
} for os_id, r in sorted(registros.items())])
with pd.option_context("display.max_colwidth", 120):
    display(resumo_execucao)

In [ ]:
for os_id in ["OS-2026-0102", "OS-2026-0103", "OS-2026-0104", "OS-2026-0105"]:
    if os_id in registros:
        print("=" * 78)
        print(os_id)
        print("=" * 78)
        mostrar_parecer(registros[os_id])
        print()

## 8.4 Verificações de comportamento com o modelo real

As verificações abaixo são aplicadas sobre os registros das triagens. Além das verificações por cenário, três valem para **todas** as triagens com o modelo: parecer aprovado pela validação, limites respeitados e disponibilidade de peças coerente com o estoque — conferida em código contra a própria ferramenta de estoque.

Nos testes com o modelo real, uma reprovação é registrada na tabela de verificações para análise, sem interromper o notebook (nos testes simulados, ao contrário, uma falha interrompe a execução). Como as respostas do modelo podem variar entre execuções, é importante analisar o resultado de cada cenário.

Uma triagem interrompida por falha externa — cota esgotada, API indisponível — termina com o parecer de contingência, que não foi produzido pelo modelo. Avaliar esse parecer como se fosse do modelo daria aprovações sem mérito: por exemplo, a contingência já recomenda bloqueio quando há gatilho de segurança. Por isso as verificações de uma triagem em contingência aparecem como **contingência** e ficam fora da contagem.

In [ ]:
MAPA_DISPONIBILIDADE = {"disponivel_na_obra": "na_obra", "disponivel_em_outro_almoxarifado": "em_outro_almoxarifado",
                        "sem_estoque": "sem_estoque", "disponivel": "em_outro_almoxarifado"}


def _usou(r: dict, ferramenta: str) -> bool:
    return ferramenta in r["ferramentas_usadas"]


def _disponibilidade_coerente(r: dict) -> bool:
    """Toda peça com disponibilidade declarada confere com o estoque real."""
    eq = _localizar_equipamento(r["equipamento_id"]) if r["equipamento_id"] != "NAO_IDENTIFICADO" else None
    obra = eq["obra_id"] if eq is not None else None
    for peca in r["parecer"]["pecas_provaveis"]:
        if peca["disponibilidade"] == "nao_verificada":
            continue
        real = consultar_estoque_pecas([peca["codigo_peca"]], obra_id=obra)
        if real["status"] != "success" or MAPA_DISPONIBILIDADE[real["pecas"][0]["situacao"]] != peca["disponibilidade"]:
            return False
    return True


VERIFICACOES = [
    ("Seleção de ferramentas", "OS-2026-0101", "consultou documentação, histórico e estoque",
     lambda r: all(_usou(r, f) for f in ["buscar_documentacao_tecnica", "consultar_historico_manutencao",
                                         "consultar_estoque_pecas"])),
    ("Seleção de ferramentas", "OS-2026-0103", "consultou o cadastro para tentar identificar o equipamento",
     lambda r: _usou(r, "consultar_equipamento")),
    ("Seleção de ferramentas", "OS-2026-0104", "consultou o plano de preventiva",
     lambda r: _usou(r, "consultar_plano_preventiva")),
    ("Seleção de ferramentas", "OS-2026-0105", "recuperou o procedimento de segurança (PRO-SEG-001)",
     lambda r: any(ref.startswith("PRO-SEG-001") for ref in r["referencias_recuperadas"])),
    ("Ausência de informação", "OS-2026-0102", "relato vago: OS devolvida com perguntas, sem confiança alta",
     lambda r: r["status_triagem"] == "devolvida" and r["parecer"]["perguntas_ao_solicitante"]
     and r["parecer"]["confianca"] != "alta"),
    ("Ausência de informação", "OS-2026-0103", "equipamento inexistente: não identificado e OS devolvida",
     lambda r: r["status_triagem"] == "devolvida" and r["equipamento_id"] == "NAO_IDENTIFICADO"),
    ("Ausência de informação", "OS-2026-0106", "OS sem relato rejeitada sem chamar o modelo",
     lambda r: r["status_triagem"] == "rejeitada" and r["rodadas"] == 0),
    ("Conclusão apropriada", "OS-2026-0101", "falha recorrente reconhecida",
     lambda r: r["parecer"]["falha_recorrente"] == "sim"),
    ("Conclusão apropriada", "OS-2026-0104", "revisão preventiva vencida reconhecida",
     lambda r: r["parecer"]["situacao_preventiva"] == "vencida"),
    ("Conclusão apropriada", "OS-2026-0105", "bloqueio e prioridade P1 recomendados",
     lambda r: r["recomendacao_operacional"] == "bloquear_ate_inspecao" and r["prioridade"] == "P1"),
    ("Conclusão apropriada", "OS-2026-0105", "bloqueio concluído pelo próprio modelo, sem precisar do guardrail",
     lambda r: not r["sobrescritas"]),
]
for os_id in ["OS-2026-0101", "OS-2026-0102", "OS-2026-0103", "OS-2026-0104", "OS-2026-0105"]:
    VERIFICACOES += [
        ("Conclusão apropriada", os_id, "parecer aprovado pela validação, sem revisão humana",
         lambda r: not r["revisao_humana_obrigatoria"]),
        ("Respeito aos limites", os_id, f"até {MAX_RODADAS} rodadas e {MAX_CHAMADAS_FERRAMENTA} ferramentas",
         lambda r: r["rodadas"] <= MAX_RODADAS and r["chamadas_ferramenta"] <= MAX_CHAMADAS_FERRAMENTA),
        ("Ausência de informação", os_id, "disponibilidade de peças coerente com o estoque",
         _disponibilidade_coerente),
    ]

linhas = []
for categoria, os_id, descricao, verificacao in VERIFICACOES:
    registro = registros.get(os_id)
    if registro is None:
        resultado = "não executada"
    elif registro["finalizacao"] == "contingencia":
        resultado = "contingência"
    else:
        try:
            resultado = "OK" if verificacao(registro) else "FALHOU"
        except Exception as erro:
            resultado = f"FALHOU ({type(erro).__name__})"
    linhas.append({"categoria": categoria, "OS": os_id, "verificação": descricao, "resultado": resultado})

tabela_verificacoes = pd.DataFrame(linhas).sort_values(["categoria", "OS"], kind="stable")
with pd.option_context("display.max_colwidth", 90):
    display(tabela_verificacoes.reset_index(drop=True))

avaliadas = tabela_verificacoes[~tabela_verificacoes["resultado"].isin(["não executada", "contingência"])]
aprovadas = (avaliadas["resultado"] == "OK").sum()
fora = tabela_verificacoes["resultado"].value_counts()
print(f"Verificações com o modelo real: {aprovadas}/{len(avaliadas)} aprovadas"
      + "".join(f" · {fora[r]} {r}" for r in ["contingência", "não executada"] if r in fora))
print(llm.resumo())

## 8.5 A fila de manutenção

O resultado final, como o planejador o recebe: uma linha por OS, ordenada por prioridade, com a recomendação, as fontes e o que precisa de atenção humana.

In [ ]:
fila = pd.read_csv(ARQUIVO_FILA, keep_default_na=False)
fila = fila.sort_values(["prioridade", "os_id"], key=lambda c: c.replace("", "P9") if c.name == "prioridade" else c)
with pd.option_context("display.max_colwidth", 60):
    display(fila[["os_id", "status_triagem", "equipamento_id", "prioridade", "recomendacao_operacional",
                  "falha_recorrente", "situacao_preventiva", "equipamento_substituto",
                  "revisao_humana_obrigatoria"]])

---
# 9. Conclusão

## 9.1 Resumo desta execução

In [ ]:
status_por_os = pd.Series({os_id: r["status_triagem"] for os_id, r in registros.items()})
print("Triagens nesta execução:")
for status, quantidade in status_por_os.value_counts().items():
    print(f"  {status:<12} {quantidade}  ({', '.join(sorted(status_por_os[status_por_os == status].index))})")
print(f"\nRevisão humana obrigatória: "
      f"{sum(r['revisao_humana_obrigatoria'] for r in registros.values())} de {len(registros)} triagens")
print(f"Sobrescritas do guardrail: {sum(len(r['sobrescritas']) for r in registros.values())} · "
      f"erros internos de ferramenta registrados no log: {len(LOG_RUNTIME)}")
print(f"Testes simulados: {sum(t['resultado'] == 'OK' for t in RESULTADOS_SIMULADOS)}/{len(RESULTADOS_SIMULADOS)} aprovados")
print(f"Uso da API — {llm.resumo()}")
print(f"Fila de manutenção gravada em: {ARQUIVO_FILA.resolve()}")

## 9.2 Requisitos da entrega e onde estão atendidos

| Requisito do enunciado | Onde está no notebook |
|---|---|
| Caracterização do caso e do problema | Seções 1.1 a 1.5 |
| Justificativa da abordagem agêntica, frente a uma automação determinística | Seção 1.6 |
| Agente com ciclo de decisão, ação e observação inspirado no ReAct | Seção 7.6 |
| Acesso a conhecimento quando necessário | Seções 5 e 6 |
| Múltiplas ferramentas integradas | Seção 6 |
| Validação de entradas e retornos | Seções 4.2, 6.4, 7.2 e 7.5 |
| Tratamento de erros | Seções 3.4, 6.1 e 6.4 |
| Limites de execução | Seções 7.1 e 7.6 |
| Estratégias de recuperação diante de falhas | Seções 3.4 e 7.6 |
| Testes de comportamento | Seção 8 |
| Notebook executável no Colab só com a chave da API | Seções 3.3, 4.1 e 5.1 |

## 9.3 Ajustes feitos a partir das execuções

Parte do desenho resultou de problemas observados ao executar o agente com o modelo real:

| Observação | Ajuste |
|---|---|
| Uma pergunta genérica sobre bloqueio recuperou o documento certo, mas não a seção certa (5.5) | A descrição da ferramenta de documentação passou a orientar perguntas específicas, uma por assunto |
| O modelo buscou três vezes os códigos de peça, que estavam no trecho já recebido, mas cortados pelo limite de tamanho (6.6) | O limite do trecho passou a cobrir a maior seção da base, com um teste de regressão |
| O Gemini 3 rejeitou o papel `tool` usado nos notebooks das aulas | Os resultados de ferramenta passaram a voltar com papel `user` |
| Erros 503, timeouts de mais de dois minutos e cota diária esgotada ocorreram durante o desenvolvimento | Retry com espera crescente, tratamento de timeout como falha transitória, detecção de cota diária e parecer de contingência (3.4 e 7.6) |
| Numa execução com o modelo sobrecarregado, as tentativas repetidas de erros 503 também consumiram a cota diária | Três tentativas por chamada, com esperas de 30 s e 60 s (3.4) |
| Em outras execuções, o `gemini-3.6-flash` respondeu 503 a todas as chamadas, enquanto outros modelos Flash da mesma chave respondiam normalmente | Lista de modelos de reserva: o notebook passa para o próximo modelo quando o atual fica indisponível ou esgota a cota (3.2 e 3.4) |
| O parecer da OS-0101 ignorou um substituto disponível em outra obra e listou uma peça já trocada (7.8) | As instruções passaram a tratar os dois casos explicitamente |
| Pareceres de contingência chegaram a ser aprovados pelas verificações escritas para o modelo (8.4) | Triagens interrompidas passaram a ser separadas e não contam como acerto |

## 9.4 Limitações conhecidas

- **Escala dos dados.** São 14 equipamentos, 7 documentos curtos e 6 ordens de serviço. Com centenas de documentos — os manuais completos dos fabricantes, por exemplo —, o chunking por seção, o top-3 e a busca puramente vetorial precisariam ser revistos; busca híbrida (lexical + vetorial) e reordenação seriam candidatas naturais.
- **Avaliação limitada.** Cinco cenários verificam propriedades estruturais do parecer, não a correção técnica do diagnóstico. Uma avaliação completa exigiria um conjunto maior de OS com gabarito definido por especialistas em manutenção.
- **Não determinismo.** O `gemini-3.6-flash` não garante controle de temperatura, e a mesma OS pode seguir caminhos diferentes entre execuções. O desenho mitiga isso com schema, validação e verificações estruturais, mas não elimina a variação.
- **Gatilhos de segurança por palavras-chave.** A camada reativa é simples e auditável, mas pode errar nos dois sentidos: deixar passar um relato escrito de forma inesperada ou disparar num uso incidental da palavra "freio". Ela complementa o julgamento do modelo e a decisão humana; não os substitui.
- **Recorrência por sistema.** A contagem considera todas as corretivas do mesmo sistema em 90 dias, sem distinguir causas diferentes dentro do sistema. Essa é a regra da política fictícia, não necessariamente a melhor prática.
- **Sem memória entre triagens.** Cada OS é triada de forma independente. Duas OS do mesmo equipamento abertas no mesmo dia não se enxergam.
- **Dependência da disponibilidade da API.** No nível gratuito, cerca de 20 chamadas de geração por dia limitam o volume de testes com o modelo real, e períodos de sobrecarga do modelo (erro 503) podem interromper triagens mesmo com as novas tentativas. Os modelos de reserva reduzem esse risco, mas a troca no meio de uma triagem a encerra em contingência, e os modelos de reserva podem produzir pareceres diferentes dos do modelo principal. Em produção, a camada de acesso ao modelo continuaria a mesma, com outra cota.

## 9.5 Próximas entregas

Como próximos passos, seria possível testar outras formas de organizar o agente e ampliar os cenários de avaliação. Nenhuma das possibilidades abaixo está implementada nesta entrega:

| Tema das próximas entregas | Possibilidade |
|---|---|
| **Orquestração com framework (LangGraph)** | O laço da seção 7.6 poderia ser representado como um grafo, com nós para percepção, decisão, execução de ferramentas, validação, guardrail e ação, usando o `EstadoTriagem` como estado do grafo. As ferramentas, a camada de acesso ao modelo e os testes simulados poderiam ser reaproveitados |
| **Sistema multiagente** | Outra possibilidade seria dividir a análise entre agentes especializados em diagnóstico técnico, peças e frota e segurança, com um agente responsável por reunir os resultados |
| **Governança, avaliação e human-in-the-loop** | A recomendação de bloqueio poderia passar a exigir aprovação formal. A trajetória registrada em cada triagem (ferramentas, argumentos, status, fontes e sobrescritas) serviria de base para auditoria, e os testes poderiam evoluir para uma avaliação em lote com gabarito |